# ¿Existen perfiles sonoros que concentren canciones de alta popularidad y qué características los distinguen?

**Objetivo general:** descubrir perfiles sonoros sin utilizar popularidad para entrenar y, después, contrastar su concentración de canciones populares y sus rasgos acústicos.

K-Means y GMM se comparan antes y después de PCA, con ajuste y validación por artistas. Una vez congelados los parámetros, se identifica en entrenamiento un perfil candidato por su tasa de popularidad >=70. La nueva evaluación del objetivo usa otras 2.000 canciones de artistas no utilizados antes. Se reportan tasas, lift, contraste frente al resto e intervalos por artistas, además de sensibilidad a 60/80.

La decisión de estudiar popularidad responde al objetivo original y está informada por análisis previos. El protocolo se fija antes de este nuevo cálculo, pero no es un prerregistro externo. No se buscan ni se garantizan causas del éxito.

Ejecutar todas las celdas desde esta carpeta. El código completo de ambos módulos aparece a continuación. Se declara asistencia de ChatGPT/Codex; completar autores y revisar los resultados antes de entregar.


### Ampliación actual
La entrega conserva la referencia de dos grupos y añade perfiles más detallados y análisis posterior por familias de género. Se recuperan todas las etiquetas por canción y se utiliza otra evaluación adicional. Las diferencias favorables y desfavorables se conservan con cobertura, captura e incertidumbre.

## Método obligatorio: clustering y PCA

In [1]:
"""Núcleo no supervisado con protocolo documentado y contraste posterior de popularidad."""
from pathlib import Path
import hashlib
import importlib.metadata
import itertools
import json
import platform
import time
import warnings
import html
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
OUT = ROOT / 'resultados'
PROTOCOL = json.loads((ROOT/'protocolo_mejora.json').read_text('utf-8'))
FEATURES = PROTOCOL['features']
SEED = PROTOCOL['seed']


Matplotlib is building the font cache; this may take a moment.


In [2]:
def dump(name, value):
    (OUT/name).write_text(json.dumps(value,ensure_ascii=False,indent=2,default=lambda v:v.item() if hasattr(v,'item') else str(v)),encoding='utf-8')


In [3]:
def table(frame):
    frame=frame.copy()
    for c in frame:
        frame[c]=frame[c].map(lambda x:f'{x:.5f}' if isinstance(x,(float,np.floating)) and np.isfinite(x) else ('—' if pd.isna(x) else str(x)))
    return '\n'.join(['| '+' | '.join(map(str,frame.columns))+' |','| '+' | '.join(['---']*len(frame.columns))+' |']+['| '+' | '.join(str(x).replace('|','/') for x in row)+' |' for row in frame.itertuples(index=False,name=None)])


In [4]:
def figure(name):
    plt.tight_layout(); plt.savefig(OUT/'figuras'/f'{name}.png',dpi=140,bbox_inches='tight'); plt.close()


In [5]:
def artists(value):
    return {v.strip().casefold() for v in str(value).split(';') if v.strip() and v.strip().casefold()!='nan'}


In [6]:
def artist_groups(frame):
    parent=list(range(len(frame))); owner={}
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]];i=parent[i]
        return i
    for i,value in enumerate(frame.artists):
        for a in sorted(artists(value)):
            if a in owner:
                # Representante canónico: no depender del orden aleatorio de sets.
                left,right=find(i),find(owner[a])
                parent[max(left,right)]=min(left,right)
            else: owner[a]=i
    return np.array([find(i) for i in range(len(frame))])


In [7]:
def artist_union(frame):
    return set().union(*(artists(v) for v in frame.artists))


In [8]:
def clean_data(frame):
    frame=frame.dropna(subset=['track_id']).drop_duplicates('track_id',keep='first').copy()
    for col in FEATURES+['popularity']:
        frame[col]=pd.to_numeric(frame[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for col in [c for c in FEATURES if c not in ['loudness','tempo','duration_ms']]:
        frame.loc[~frame[col].between(0,1),col]=np.nan
    for col in ['tempo','duration_ms']:
        frame.loc[frame[col]<=0,col]=np.nan
    frame.loc[~frame.popularity.between(0,100),'popularity']=np.nan
    return frame.loc[frame[FEATURES].notna().any(axis=1)].reset_index(drop=True)


In [9]:
def signatures(frame):
    return pd.util.hash_pandas_object(frame[FEATURES].astype('float64').round(7),index=False).to_numpy()


In [10]:
def splits():
    raw=pd.read_csv(ROOT/'data/raw/spotify_tracks.csv')
    clean=clean_data(raw)
    prior_train=clean_data(pd.read_csv(ROOT/'data/referencia_anterior/train.csv'))
    prior_test=clean_data(pd.read_csv(ROOT/'data/referencia_anterior/test.csv'))
    previous=pd.concat([prior_train,prior_test],ignore_index=True)
    groups=artist_groups(prior_train)
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=PROTOCOL['validation_group_fraction'],random_state=SEED).split(prior_train,groups=groups))
    fit=prior_train.iloc[a].reset_index(drop=True); validation=prior_train.iloc[b].reset_index(drop=True)
    previous_artists=artist_union(previous)
    mask=~clean.track_id.isin(previous.track_id)
    mask &= clean.artists.map(lambda v:bool(artists(v)) and not bool(artists(v)&previous_artists))
    mask &= ~np.isin(signatures(clean),signatures(previous))
    eligible=clean.loc[mask]
    if len(eligible)<2000: raise ValueError('No existen 2000 canciones reservables con los filtros predefinidos')
    holdout=eligible.sample(n=2000,random_state=PROTOCOL['holdout_seed']).reset_index(drop=True)
    assert not artist_union(fit)&artist_union(validation)
    assert not artist_union(previous)&artist_union(holdout)
    assert not set(previous.track_id)&set(holdout.track_id)
    assert not set(signatures(previous))&set(signatures(holdout))
    audit=dict(raw_rows=len(raw),unique_rows=len(clean),development=len(prior_train),inner_fit=len(fit),validation=len(validation),holdout=len(holdout),eligible_holdout=len(eligible),previous_test_reused_for_selection=False,shared_artists_fit_validation=0,shared_artists_previous_holdout=0,shared_exact_audio_previous_holdout=0,raw_missing=raw[FEATURES].isna().sum().to_dict(),training_missing=prior_train[FEATURES].isna().sum().to_dict(),dataset_sha256=hashlib.sha256((ROOT/'data/raw/spotify_tracks.csv').read_bytes()).hexdigest())
    for name,frame in [('entrenamiento',prior_train),('validacion_interna',validation),('ajuste_interno',fit),('prueba_reservada',holdout)]:frame.to_csv(OUT/f'{name}.csv',index=False)
    dump('auditoria.json',audit)
    return prior_train,fit,validation,holdout,audit


In [11]:
def pipeline(preprocessing):
    if preprocessing=='standard':return make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
    return make_pipeline(SimpleImputer(strategy='median'),PowerTransformer(method='yeo-johnson',standardize=True))


In [12]:
def estimator(family,k,covariance='NA',seed=SEED):
    if family=='KMeans':return KMeans(n_clusters=int(k),n_init=20,random_state=seed)
    return GaussianMixture(n_components=int(k),covariance_type=covariance,n_init=3,max_iter=500,reg_covar=1e-5,random_state=seed)


In [13]:
def fit_model(est,X):
    t=time.perf_counter()
    with warnings.catch_warnings(record=True) as records:
        warnings.simplefilter('always',ConvergenceWarning);est.fit(X)
    return time.perf_counter()-t,bool(getattr(est,'converged_',True)),len(records)


In [14]:
def silhouette(X,labels,limit=1200):
    ix=np.random.default_rng(SEED).choice(len(X),min(limit,len(X)),replace=False)
    labs=labels[ix]
    return float(silhouette_score(X[ix],labs)) if 1<len(np.unique(labs))<len(ix) else np.nan


In [15]:
def metrics(X,labels):
    if len(np.unique(labels))<2:return dict(silhouette=np.nan,davies_bouldin=np.nan,calinski_harabasz=np.nan)
    return dict(silhouette=silhouette(X,labels,2000),davies_bouldin=float(davies_bouldin_score(X,labels)),calinski_harabasz=float(calinski_harabasz_score(X,labels)))


In [16]:
def choose(frame):
    ok=frame.loc[frame.eligible].copy()
    if ok.empty:raise RuntimeError('Ninguna configuración cumple separación, convergencia y tamaño mínimo')
    near=ok.loc[ok.val_sil_common>=ok.val_sil_common.max()-PROTOCOL['silhouette_tolerance']]
    return near.sort_values(['k','bic'],na_position='last').iloc[0]


In [17]:
def search(fit,validation):
    reference=pipeline('standard').fit(fit[FEATURES])
    common_train=reference.transform(fit[FEATURES]);common_val=reference.transform(validation[FEATURES])
    rows=[];selected=[]
    for prep in PROTOCOL['preprocessing']:
        transform=pipeline(prep).fit(fit[FEATURES]);X=transform.transform(fit[FEATURES]);V=transform.transform(validation[FEATURES])
        pca=PCA(n_components=PROTOCOL['pca_variance'],svd_solver='full').fit(X)
        for space,A,B in [('original',X,V),('pca',pca.transform(X),pca.transform(V))]:
            for family in ['KMeans','GMM']:
                current=[]
                for k in PROTOCOL['k_grid']:
                    for cov in (['NA'] if family=='KMeans' else PROTOCOL['gmm_covariances']):
                        est=estimator(family,k,cov);seconds,converged,nwarnings=fit_model(est,A)
                        lt=est.predict(A);lv=est.predict(B)
                        sizes_t=np.bincount(lt,minlength=k)/len(lt);sizes_v=np.bincount(lv,minlength=k)/len(lv)
                        score=silhouette(common_val,lv)
                        row=dict(preprocessing=prep,model=family,space=space,k=k,covariance=cov,converged=converged,warnings=nwarnings,fit_seconds=seconds,train_sil_native=silhouette(A,lt),val_sil_native=silhouette(B,lv),val_sil_common=score,min_train_fraction=float(sizes_t.min()),min_val_fraction=float(sizes_v.min()),bic=float(est.bic(A)) if family=='GMM' else np.nan,aic=float(est.aic(A)) if family=='GMM' else np.nan,inertia=float(est.inertia_) if family=='KMeans' else np.nan,validation_loglik=float(est.score(B)) if family=='GMM' else np.nan)
                        row['eligible']=bool(converged and np.isfinite(score) and sizes_t.min()>=PROTOCOL['min_cluster_fraction'] and sizes_v.min()>=PROTOCOL['min_cluster_fraction'])
                        current.append(row);rows.append(row)
                frame=pd.DataFrame(current);best=choose(frame);selected.append(best.to_dict())
                frame.to_csv(OUT/f'busqueda_{prep}_{family}_{space}.csv',index=False)
                fig,axs=plt.subplots(1,2,figsize=(11,3.8))
                for cov,g in frame.groupby('covariance'):
                    axs[0].plot(g.k,g.val_sil_common,'o-',label=cov);axs[1].plot(g.k,g.bic if family=='GMM' else g.inertia,'o-',label=cov)
                axs[0].set(xlabel='Número de grupos',ylabel='Silhouette validación, geometría común',title=f'{family}: separación');axs[1].set(xlabel='Número de grupos',ylabel='BIC (menor mejor)' if family=='GMM' else 'Inercia',title='Densidad / codo');axs[0].legend();axs[1].legend();figure(f'busqueda_{prep}_{family}_{space}')
                print(f'{prep}/{family}/{space}: K={best.k}, cov={best.covariance}, val común={best.val_sil_common:.4f}',flush=True)
    all_rows=pd.DataFrame(rows);selected=pd.DataFrame(selected)
    all_rows.to_csv(OUT/'busqueda_completa.csv',index=False);selected.to_csv(OUT/'seleccion_por_preparacion.csv',index=False)
    summary=selected.groupby('preprocessing').val_sil_common.mean()
    prep='yeo_johnson' if summary['yeo_johnson']>summary['standard']+PROTOCOL['preprocessing_min_gain'] else 'standard'
    frozen=dict(protocol_sha256=hashlib.sha256((ROOT/'protocolo_mejora.json').read_bytes()).hexdigest(),preprocessing=prep,mean_validation_silhouette=summary.to_dict(),selected=selected.loc[selected.preprocessing==prep].replace({np.nan:None}).to_dict('records'),selection_uses_popularity=False,selection_uses_holdout=False)
    dump('seleccion_congelada.json',frozen)
    return frozen,all_rows,selected


In [18]:
def final_experiment(train,holdout,frozen):
    prep=frozen['preprocessing'];before=time.perf_counter()
    transform=pipeline(prep).fit(train[FEATURES]);X=transform.transform(train[FEATURES]);T=transform.transform(holdout[FEATURES])
    reference=pipeline('standard').fit(train[FEATURES]);common_train=reference.transform(train[FEATURES]);common_test=reference.transform(holdout[FEATURES])
    pca=PCA(n_components=PROTOCOL['pca_variance'],svd_solver='full').fit(X);Z=pca.transform(X);W=pca.transform(T)
    preprocess_seconds=time.perf_counter()-before
    spectrum=PCA(svd_solver='full').fit(X)
    variance=pd.DataFrame(dict(component=np.arange(1,11),ratio=spectrum.explained_variance_ratio_,cumulative=np.cumsum(spectrum.explained_variance_ratio_)));variance.to_csv(OUT/'varianza_pca.csv',index=False)
    loads=pd.DataFrame(pca.components_.T,index=FEATURES,columns=[f'PC{i+1}' for i in range(pca.n_components_)]);loads.to_csv(OUT/'cargas_pca.csv')
    plt.figure(figsize=(7,4));plt.plot(variance.component,variance.cumulative,'o-');plt.axhline(.9,color='red',ls='--',label='90 %');plt.axvline(pca.n_components_,color='gray',ls=':');plt.xlabel('Componentes');plt.ylabel('Varianza acumulada');plt.title('Elección de PCA solo con entrenamiento');plt.legend();figure('varianza_pca')
    models={};labels={};rows=[];stability=[];profiles=[];pops=[];unscaled=[];timings=[]
    for cfg in frozen['selected']:
        family,space=cfg['model'],cfg['space'];key=family+'_'+space
        A,B=(X,T) if space=='original' else (Z,W)
        est=estimator(family,cfg['k'],cfg['covariance']);seconds,converged,_=fit_model(est,A)
        if not converged:raise RuntimeError(f'No converge {key} al reajustar')
        lt=est.predict(A);tick=time.perf_counter();lv=est.predict(B);predict_seconds=time.perf_counter()-tick
        models[key]=est;labels[key]=(lt,lv)
        for split,matrix,common,lab in [('train',A,common_train,lt),('test',B,common_test,lv)]:
            ari=adjusted_rand_score(labels[family+'_original'][0 if split=='train' else 1],lab) if space=='pca' else np.nan
            for geometry,evaluation in [('native',matrix),('reference_original',common)]:
                rows.append(dict(model=family,space=space,preprocessing=prep,split=split,evaluation_space=geometry,hyperparameters=json.dumps(dict(k=cfg['k'],covariance=cfg['covariance'])),groups=len(np.unique(lab)),noise_pct=0.,fit_seconds=seconds,predict_seconds=predict_seconds,ari_vs_original=ari,**metrics(evaluation,lab)))
        # Las repeticiones no seleccionan ni vuelven a ajustar hiperparámetros.
        for kind in ['seed','subsample']:
            repeated=[]
            for seed in [42,7,21,123]:
                idx=np.random.default_rng(seed).choice(len(A),int(.8*len(A)),replace=False) if kind=='subsample' else np.arange(len(A))
                e=estimator(family,cfg['k'],cfg['covariance'],seed);_,ok,_=fit_model(e,A[idx])
                if not ok:raise RuntimeError(f'No converge repetición {key}/{kind}/{seed}')
                repeated.append(e.predict(B))
            for i,j in itertools.combinations(range(4),2):stability.append(dict(model=family,space=space,kind=kind,run_a=i,run_b=j,ari=float(adjusted_rand_score(repeated[i],repeated[j]))))
        assigned=pd.concat([train.assign(split='train',cluster=lt),holdout.assign(split='test',cluster=lv)],ignore_index=True)
        assigned.to_csv(OUT/f'asignaciones_{key}.csv',index=False)
        med=train.assign(cluster=lt).groupby('cluster')[FEATURES].median();med.to_csv(OUT/f'perfiles_{key}.csv')
        sizes=pd.Series(lt).value_counts()
        for c,r in med.iterrows():profiles.append(dict(model=family,space=space,cluster=int(c),n=int(sizes[c]),**r.to_dict()))
        for split,g in assigned.groupby('split'):
            for c,h in g.groupby('cluster'):pops.append(dict(model=family,space=space,split=split,cluster=int(c),n=len(h),median=float(h.popularity.median()),q25=float(h.popularity.quantile(.25)),q75=float(h.popularity.quantile(.75))))
        plt.figure(figsize=(9,4));plt.imshow(((med-train[FEATURES].mean())/train[FEATURES].std()).to_numpy(),aspect='auto',cmap='RdBu_r',vmin=-2,vmax=2);plt.xticks(range(10),FEATURES,rotation=75);plt.yticks(range(len(med)),med.index);plt.colorbar(label='Mediana respecto a media global (DE)');plt.title('Perfiles: '+key);figure('perfiles_'+key)
        plt.figure(figsize=(7,4));sc=plt.scatter(Z[:,0],Z[:,1],c=lt,cmap='tab20',s=4,alpha=.45);plt.colorbar(sc,label='Grupo');plt.xlabel('PC1');plt.ylabel('PC2');plt.title(key+': proyección, no prueba de separación');figure('mapa_'+key)
        # Benchmark corto repetido, sin búsqueda ni preprocesamiento.
        for repeat in range(3):
            e=estimator(family,cfg['k'],cfg['covariance']);elapsed,ok,_=fit_model(e,A)
            tick=time.perf_counter();e.predict(B);prediction=time.perf_counter()-tick
            timings.append(dict(model=family,space=space,repeat=repeat,fit_seconds=elapsed,predict_seconds=prediction,converged=ok))
    # Control sin escalar en los dos modelos originales, evaluado en geometría común.
    U=SimpleImputer(strategy='median').fit_transform(train[FEATURES])
    for family in ['KMeans','GMM']:
        cfg=next(c for c in frozen['selected'] if c['model']==family and c['space']=='original')
        e=estimator(family,cfg['k'],cfg['covariance']);_,ok,_=fit_model(e,U)
        if not ok:raise RuntimeError('No converge control sin escalar: '+family)
        labs=e.predict(U);unscaled.append(dict(model=family,ari_scaled_unscaled=float(adjusted_rand_score(labels[family+'_original'][0],labs)),silhouette_scaled_common=silhouette(common_train,labels[family+'_original'][0],2000),silhouette_unscaled_labels_common=silhouette(common_train,labs,2000)))
    for filename,value in [('comparativa',rows),('estabilidad',stability),('perfiles',profiles),('popularidad',pops),('control_escalado',unscaled),('tiempos_repetidos',timings)]:pd.DataFrame(value).to_csv(OUT/f'{filename}.csv',index=False)
    stats=pd.DataFrame(dict(skew_raw=train[FEATURES].skew(),skew_selected=pd.DataFrame(X,columns=FEATURES).skew(),iqr_outlier_fraction=((train[FEATURES]<train[FEATURES].quantile(.25)-1.5*(train[FEATURES].quantile(.75)-train[FEATURES].quantile(.25)))|(train[FEATURES]>train[FEATURES].quantile(.75)+1.5*(train[FEATURES].quantile(.75)-train[FEATURES].quantile(.25)))).mean()));stats.to_csv(OUT/'asimetria_atipicos.csv')
    train[FEATURES].describe().T.to_csv(OUT/'descriptiva.csv')
    train[FEATURES].hist(figsize=(12,8),bins=35);plt.suptitle('Distribuciones de entrenamiento en unidades originales');figure('distribuciones')
    corr=train[FEATURES].corr();corr.to_csv(OUT/'correlaciones.csv');plt.figure(figsize=(9,7));plt.imshow(corr,vmin=-1,vmax=1,cmap='RdBu_r');plt.xticks(range(10),FEATURES,rotation=90);plt.yticks(range(10),FEATURES);plt.colorbar(label='Pearson r');plt.title('Correlaciones acústicas');figure('correlaciones')
    summary=dict(preprocessing=prep,pca_components=int(pca.n_components_),pca_variance=float(pca.explained_variance_ratio_.sum()),pc1_pc2_variance=float(pca.explained_variance_ratio_[:2].sum()),preprocess_seconds=preprocess_seconds)
    dump('transformacion.json',summary)
    return pd.DataFrame(rows),pd.DataFrame(stability),pd.DataFrame(profiles),pd.DataFrame(pops),loads,labels,common_train,common_test,summary


In [19]:
def listening_example(train,test,labels,common_train,common_test,config):
    a,b=labels[config];low=np.flatnonzero(train.popularity.to_numpy()<=30);high=np.flatnonzero(test.popularity.to_numpy()>=70);pairs=[]
    for qi in high:
        pool=low[a[low]==b[qi]]
        if not len(pool):continue
        distance=np.linalg.norm(common_train[pool]-common_test[qi],axis=1);i=int(pool[np.argmin(distance)])
        pairs.append(dict(query_index=int(qi),reference_index=i,distance=float(distance.min()),high_song=str(test.iloc[qi].track_name),high_artist=str(test.iloc[qi].artists),high_id=str(test.iloc[qi].track_id),high_popularity=int(test.iloc[qi].popularity),low_song=str(train.iloc[i].track_name),low_artist=str(train.iloc[i].artists),low_id=str(train.iloc[i].track_id),low_popularity=int(train.iloc[i].popularity),configuration=config))
    frame=pd.DataFrame(pairs).sort_values(['distance','high_id']) if pairs else pd.DataFrame()
    frame.to_csv(OUT/'ejemplos_escucha.csv',index=False)
    if len(frame):
        r=frame.iloc[0];comparison=pd.DataFrame({'variable':FEATURES,'cancion_popular':test.iloc[int(r.query_index)][FEATURES].to_numpy(),'referencia_menor_popularidad':train.iloc[int(r.reference_index)][FEATURES].to_numpy()});comparison.to_csv(OUT/'comparacion_escucha.csv',index=False)
    return frame


In [20]:
def render_report(blocks):
    result=[]
    for b in blocks:
        if b.startswith('!['):
            alt,url=b[2:].split('](',1);result.append('<figure><img src="'+html.escape(url[:-1])+'" alt="'+html.escape(alt)+'"><figcaption>'+html.escape(alt)+'</figcaption></figure>')
        elif b.startswith('| '):
            lines=b.splitlines();row=lambda l,t:'<tr>'+''.join('<'+t+'>'+html.escape(v.strip())+'</'+t+'>' for v in l.strip('|').split('|'))+'</tr>'
            result.append('<div class="table"><table>'+row(lines[0],'th')+''.join(row(l,'td') for l in lines[2:])+'</table></div>')
        elif b.startswith('#'):
            level=len(b)-len(b.lstrip('#'));result.append(f'<h{level}>'+html.escape(b[level:].strip())+f'</h{level}>')
        else:
            escaped=html.escape(b)
            escaped=re.sub(r'(https://[^\s<>]+)',r'<a href="\1" target="_blank" rel="noopener noreferrer">\1</a>',escaped)
            result.append('<p>'+escaped+'</p>')
    return '<!doctype html><html lang="es"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Segmentación musical — Informe mejorado</title><style>body{max-width:1120px;margin:40px auto;padding:24px;font:16px/1.65 system-ui;color:#183044}h1,h2,h3{color:#124e68}img{max-width:100%}figure{margin:24px 0}figcaption{font-size:12px}.table{overflow:auto}table{border-collapse:collapse;font-size:12px}th,td{padding:8px;border:1px solid #c8d7df}th{background:#eaf2f6}p{white-space:pre-line}@media print{figure,table{break-inside:avoid}}</style><main>'+''.join(result)+'</main></html>'


In [21]:
def report(audit,frozen,searches,selected,comparison,stable,profiles,pops,loads,summary,examples):
    common=comparison.query("evaluation_space=='reference_original'")
    selection=pd.DataFrame(frozen['selected']);winner=selection.sort_values('val_sil_common',ascending=False).iloc[0];key=winner.model+'_'+winner.space
    benchmark=pd.read_csv(OUT/'tiempos_repetidos.csv').groupby(['model','space'])[['fit_seconds','predict_seconds']].agg(['median','min','max']).reset_index();benchmark.columns=['_'.join(c).rstrip('_') if isinstance(c,tuple) else c for c in benchmark.columns]
    stab=stable.groupby(['model','space','kind']).ari.agg(['mean','min','max']).reset_index()
    correlations=pd.read_csv(OUT/'correlaciones.csv',index_col=0);pairs=sorted([(a,b,float(correlations.loc[a,b])) for i,a in enumerate(FEATURES) for b in FEATURES[i+1:]],key=lambda x:abs(x[2]),reverse=True)[:3]
    blocks=['# Segmentación musical: perfiles fiables, sin prometer éxitos',
        'Comparación de K-Means y GMM antes y después de PCA · ISW-222 / Bloque 3. Autor(es) y fecha de entrega: completar antes de presentar.',
        '## Resumen',f'Se reutilizan {audit["development"]} canciones de desarrollo. La selección usa {audit["inner_fit"]} para ajuste y {audit["validation"]} para validación, separados por componentes de artistas colaboradores. La prueba adicional tiene {audit["holdout"]} canciones fuera de la muestra anterior y sin artistas compartidos. Preparación elegida: {summary["preprocessing"]}. PCA conserva {summary["pca_components"]} componentes ({summary["pca_variance"]:.2%}). Recomendación fijada con validación: {key}. Popularidad nunca interviene en ajustes ni selección.',
        '## 1. Problema y aporte propio',
        '¿Cómo organizar un catálogo por perfiles sonoros interpretables y estables, y cuánto cambian al aplicar PCA? Los grupos permiten filtrar referencias acústicas para exploración musical y producción. No hay etiquetas verdaderas de perfil; género, nombre, artista y popularidad se excluyen del vector de entrenamiento. Se investiga descriptivamente si perfiles parecidos pueden alcanzar popularidades distintas.',
        'El aporte es comparar preparación, familias y reducción bajo el mismo criterio, controlar artistas compartidos y reservar una prueba adicional. Una mejora de métricas no demuestra calidad artística ni una fórmula de éxito.',
        '## 2. Datos y preparación sin fuga',
        'Fuente: Pandya, Spotify Tracks Dataset, Hugging Face. Licencia declarada en su ficha: bsd; no se atribuye una variante BSD no especificada. Se utilizan datos tabulares y no se distribuye audio. La popularidad es la registrada en el archivo, no reproducciones actuales.',
        table(pd.DataFrame([dict(variable=f,unidad='ms' if f=='duration_ms' else 'BPM' if f=='tempo' else 'dB' if f=='loudness' else '0–1',faltantes_fuente=audit['raw_missing'][f],faltantes_train=audit['training_missing'][f]) for f in FEATURES])),
        table(pd.DataFrame([dict(etapa=k,n=audit[k]) for k in ['raw_rows','unique_rows','development','inner_fit','validation','eligible_holdout','holdout']])),
        'Se conserva la primera aparición por ID sin seleccionar por popularidad. Variables fuera de rango, valores no finitos y tempo/duración no positivos pasan a faltantes. Medianas y transformaciones se ajustan solo con el subconjunto de ajuste, y se reajustan en desarrollo completo una vez congeladas las elecciones.',
        'Los grupos de artistas unen colaboraciones transitivamente; dividir solo por cadena completa de artista dejaría fugas. En la prueba adicional se excluyen IDs previos, artistas individuales de toda la muestra anterior y vectores acústicos iguales tras redondeo a siete decimales. Esto no identifica todas las versiones semánticamente duplicadas ni constituye una muestra temporal o un dataset externo.',
        table(pd.read_csv(OUT/'descriptiva.csv').rename(columns={'Unnamed: 0':'variable'})),
        '![Distribuciones en unidades originales](figuras/distribuciones.png)',
        table(pd.read_csv(OUT/'asimetria_atipicos.csv').rename(columns={'Unnamed: 0':'variable'})),
        'Los extremos IQR se describen, no se eliminan automáticamente: instrumentalidad o habla elevadas pueden ser legítimas. Yeo-Johnson transforma las distribuciones marginales y después estandariza; no garantiza gaussianidad conjunta ni mejores agrupamientos. StandardScaler iguala escalas, pero no elimina asimetría.',
        '![Correlaciones de entrenamiento](figuras/correlaciones.png)',
        '; '.join(f'{a}/{b}: r={r:+.3f}' for a,b,r in pairs)+'. Las correlaciones anticipan redundancia parcial, no autorizan a descartar información sin comparar.',
        '## 3. Protocolo y selección',
        'El protocolo se escribió antes de ejecutar la nueva búsqueda y se conserva con huella SHA-256. Esta revisión es exploratoria y está informada por la versión anterior; no se presenta como prerregistro externo. No se usa la prueba adicional para seleccionar. El test anterior ya observado tampoco selecciona.',
        table(pd.DataFrame([dict(preparacion=k,silhouette_validacion_media=v) for k,v in frozen['mean_validation_silhouette'].items()])),
        'Regla común: maximizar Silhouette en validación, calculado siempre en las diez variables estandarizadas por el escalador de referencia. Se exige al menos 2 % por grupo en ajuste y validación. Dentro de 0,005 del máximo se prefiere menor K; a igualdad de K, menor BIC para GMM. Para cambiar de preparación se exige ganancia media superior a 0,005 en las cuatro configuraciones. Es una regla explícita de parsimonia, no una prueba de significación.',
        'BIC y AIC se calculan como diagnósticos de densidad, pero no se priorizan sobre la utilidad de segmentación. Este cambio de objetivo respecto a la versión anterior se declara; no prueba que la densidad GMM ahora esté mejor ajustada. BIC no se compara entre espacios o transformaciones diferentes. La geometría común favorece por definición conservar la representación estandarizada original.',
        table(selected[['preprocessing','model','space','k','covariance','val_sil_common','bic','min_train_fraction','min_val_fraction']])]
    for prep in PROTOCOL['preprocessing']:
        for family in ['KMeans','GMM']:
            for space in ['original','pca']:blocks.append(f'![Selección {prep}/{family}/{space}](figuras/busqueda_{prep}_{family}_{space}.png)')
    blocks+=['## 4. PCA e interpretación',f'Umbral fijado en 90 %: se retienen {summary["pca_components"]} componentes y {summary["pca_variance"]:.2%}. PC1+PC2 muestran solo {summary["pc1_pc2_variance"]:.2%}; no son todo el espacio del modelo.',
        '![Varianza acumulada de PCA](figuras/varianza_pca.png)',table(loads.iloc[:,:2].reset_index(names='variable'))]
    for pc in ['PC1','PC2']:
        dom=loads[pc].abs().nlargest(4).index
        blocks.append(pc+': '+', '.join(f'{v} ({loads.loc[v,pc]:+.3f})' for v in dom)+'. El componente contrasta los atributos positivos y negativos indicados. Si se selecciona Yeo-Johnson, las cargas corresponden a características transformadas, no a relaciones lineales con las unidades físicas originales. El signo global es arbitrario.')
    blocks+=['PC1 contrapone intensidad y nivel sonoro (energy y loudness) frente a carácter acústico (acousticness). En términos musicales, resume un eje de sonido más intenso/producido frente a textura más acústica; no clasifica automáticamente géneros ni calidad.',
        'PC2 contrapone valencia y bailabilidad frente a instrumentalidad y duración. Describe un eje de positividad sonora y ritmo bailable frente a formatos más largos o instrumentales. Son interpretaciones descriptivas de las cargas observadas, no emociones del oyente ni una receta de éxito.',
        '## 5. Tabla comparativa obligatoria y comparación justa',
        'Original significa diez características preparadas, sin PCA. Native usa la geometría de cada ajuste; reference_original siempre utiliza las diez características con StandardScaler de referencia. Solo estas últimas filas permiten comparar separaciones en la misma geometría. Se incluyen entrenamiento y prueba, hiperparámetros, grupos, tres métricas, tiempos y ARI. Ruido=0 porque ambos asignan todas las canciones; no equivale a detectar ausencia de anomalías.',table(comparison),
        'Silhouette: mayor mejor, cohesión frente a separación. Davies-Bouldin: menor mejor, similitud entre grupos. Calinski-Harabasz: mayor mejor, dispersión entre grupos respecto a dispersión interna. Favorecen agrupamientos compactos y no son medidas de gusto o éxito. Silhouette usa hasta 2.000 observaciones con semilla fija; las otras métricas usan el conjunto completo.',
        '## 6. Estabilidad y costo',table(stab),
        'Cuatro semillas y cuatro submuestras de 80 % por configuración; ARI entre pares sobre las mismas canciones de prueba. Transformación y PCA permanecen fijos: mide estabilidad condicional de los modelos. No se selecciona con esta prueba. La incertidumbre de toda la preparación requiere remuestrear el pipeline completo.',table(benchmark),
        f'Preparación final compartida (incluye representación de referencia y PCA): {summary["preprocess_seconds"]:.4f} s. Los tiempos repetidos de modelos excluyen búsqueda y preparación, usan un hilo numérico y no son una garantía de rendimiento en otro equipo.',
        '## 7. Interpretación y acciones por perfil']
    for cfg in frozen['selected']:
        family,space=cfg['model'],cfg['space'];name=family+'_'+space
        group=profiles.query('model==@family and space==@space')
        blocks += ['### '+name,table(group.drop(columns=['model','space'])),f'![Perfiles originales {name}](figuras/perfiles_{name}.png)',f'![Mapa PC1/PC2 {name}](figuras/mapa_{name}.png)']
        for r in group.itertuples():
            energy='intenso' if r.energy>=.65 else 'suave' if r.energy<.35 else 'intermedio';texture='acústico' if r.acousticness>=.6 else 'instrumental' if r.instrumentalness>=.5 else 'mixto'
            blocks.append(f'Grupo {r.cluster}: {energy}, {texture}, {r.tempo:.1f} BPM, {r.duration_ms/60000:.2f} min; {r.n} canciones. Acción: filtrar referencias con esta combinación y contrastarlas por escucha. No prescribe una mezcla, género verdadero o público objetivo.')
    blocks+=['### Popularidad como análisis secundario',table(pops),
        'Las medianas y cuartiles describen el archivo. Las diferencias no garantizan éxito individual ni prueban causalidad. La validación por artistas cambia la población: no se comparan sin más estas cifras con las del test anterior.',
        '## 8. Las doce respuestas de la rúbrica',
        '### 1. Escalado',table(pd.read_csv(OUT/'control_escalado.csv')),
        'Duración en milisegundos domina distancias sin escalar frente a variables 0–1. El control conserva hiperparámetros y evalúa sus etiquetas en la misma geometría común. El ARI cuantifica el cambio de partición y evita concluir por un Silhouette medido con otra escala.',
        '### 2. Elección de hiperparámetros',table(selection[['model','space','k','covariance','val_sil_common','bic']]),
        'La búsqueda y sus curvas abarcan K=2–10,12,16,20,24 y covarianzas full/diag para GMM. Se prioriza separación de validación con tamaños mínimos y parsimonia; BIC explica ajuste de densidad, distinto del objetivo de segmentación. La tabla completa conserva todos los intentos, no solo ganadores.',
        '### 3. Sesgo de las métricas',table(common.query("split=='test'")[[ 'model','space','silhouette','davies_bouldin','calinski_harabasz']]),
        'El objetivo euclidiano y el criterio favorecen a K-Means y a gaussianas compactas. Las nubes superpuestas en los mapas advierten que los perfiles pueden ser discretizaciones de un continuo. La proyección PC1/PC2 no prueba separación en todas las dimensiones; contrastar con estas métricas y no solo colores.',
        '### 4. Ruido y atípicos',table(pd.read_csv(OUT/'asimetria_atipicos.csv').rename(columns={'Unnamed: 0':'variable'})),
        'No se utilizó DBSCAN. K-Means y GMM asignan todos los puntos: ruido 0 % es una propiedad del método. Los atípicos pueden desplazar centroides y covarianzas. No se eliminan perfiles musicales válidos por IQR; se compara una transformación marginal predefinida y se describen sus límites.',
        '### 5. Componentes principales',f'{summary["pca_components"]} componentes, varianza {summary["pca_variance"]:.2%}; dos primeras dimensiones {summary["pc1_pc2_variance"]:.2%}.',table(loads.iloc[:,:2].reset_index(names='variable')),
        'Las mayores cargas absolutas de la sección 4 definen los contrastes acústicos; se interpretan con sus signos. La varianza retenida no es porcentaje de información musical ni de éxito explicado.',
        '### 6. Efecto de la reducción']
    for family in ['KMeans','GMM']:
        a=common.query("model==@family and split=='test' and space=='original'").iloc[0];b=common.query("model==@family and split=='test' and space=='pca'").iloc[0]
        blocks.append(f'{family}, prueba en geometría común: Silhouette {a.silhouette:.4f} → {b.silhouette:.4f}; Davies-Bouldin {a.davies_bouldin:.4f} → {b.davies_bouldin:.4f}; Calinski-Harabasz {a.calinski_harabasz:.2f} → {b.calinski_harabasz:.2f}. PCA puede quitar redundancia o descartar diferencias útiles; las cifras no identifican por sí solas cuál causa actuó. Diez variables no justifican invocar automáticamente maldición de dimensionalidad.')
    blocks+=['### 7. Comparación justa',table(comparison.query("split=='train'")[[ 'model','space','evaluation_space','silhouette','davies_bouldin','calinski_harabasz']]),
        'Comparar Native original con Native PCA mezcla geometrías. La conclusión de separación utiliza reference_original. Cuando las conclusiones difieren, se prioriza esta geometría declarada y se reconoce el sesgo a favor de preservarla.',
        '### 8. Cambio de asignaciones',table(common.query("space=='pca'")[[ 'model','split','ari_vs_original']]),
        'ARI=1 implica particiones equivalentes aun con etiquetas permutadas; cercano a cero es acuerdo compatible con azar. Puede ser negativo. Se compara el mismo conjunto en cada fila y no popularidad o género.',
        '### 9. Estabilidad',table(stab)]
    for space in ['original','pca']:
        s=stable.query('space==@space').groupby('model').ari.mean().sort_values(ascending=False)
        blocks.append(f'En {space}, mayor ARI medio condicional: {s.index[0]}={s.iloc[0]:.4f}; '+', '.join(f'{k}={v:.4f}' for k,v in s.iloc[1:].items())+'. K-Means puede variar por centroides iniciales; GMM añade sensibilidad de covarianzas y máximos locales. Un ARI alto no prueba que los grupos sean verdaderos.')
    blocks+=['### 10. Costo computacional',table(benchmark),
        'Con 100 veces más observaciones no se extrapolan segundos ciegamente. K-Means escala aproximadamente O(n·K·d·iteraciones); GMM full añade operaciones de covarianza O(n·K·d²) y factorización O(K·d³) por iteración. Reducir d puede ayudar a GMM, pero depende también de convergencia. Silhouette completo tiene costo cuadrático en n; mantener muestreo. Incluir ajuste/transformación PCA antes de afirmar ahorro total.',
        '### 11. Visualización no lineal',f'No se ejecutó UMAP ni t-SNE: son opcionales. PCA muestra {summary["pc1_pc2_variance"]:.2%} en dos ejes y no todo el espacio. Islas en una visualización no lineal hipotética no demostrarían grupos naturales. t-SNE altera distancias globales y densidades, depende de parámetros y no es entrada de este clustering.',
        '### 12. Recomendación final',f'Se recomienda provisionalmente {key}, elegido en validación con Silhouette común={winner.val_sil_common:.4f}; no se cambia por los resultados de prueba ni popularidad. Revisar conjuntamente las tres métricas, estabilidad, tamaños y perfiles, y reconocer que diferencias pequeñas no demuestran superioridad estadística.',
        'Limitaciones: muestra histórica de conveniencia; exclusión por artistas modifica la población y no equivale a evaluación temporal; versiones no detectadas; características agregadas omiten melodía, letra y secciones; sin evaluación de oyentes. Con más tiempo: validación temporal/externa, remuestreo de toda la preparación, protocolo humano y medidas longitudinales de exposición/promoción.',
        '## 9. Cierre con canciones reales']
    if len(examples):
        r=examples.iloc[0];blocks += [f'Hay {len(examples)} consultas de popularidad ≥70 con al menos una referencia ≤30 dentro del mismo perfil. La pareja siguiente minimiza su distancia en las diez variables estandarizadas; es un ejemplo deliberadamente seleccionado para ilustrar similitud con popularidades distintas, no prueba representativa de todas las canciones.',table(examples.head(3).drop(columns=['query_index','reference_index'])),table(pd.read_csv(OUT/'comparacion_escucha.csv')),
            'Abrir las canciones legalmente en Spotify y comprobar mediante escucha si la similitud percibida coincide con las características. No se seleccionó un tramo del audio ni se analizó estructura verso/estribillo; el dataset no contiene audio. Si al escuchar no parecen similares, explicar qué no capturan las variables y no ocultar ese resultado.',f'Canción A: https://open.spotify.com/track/{r.high_id}\nCanción B: https://open.spotify.com/track/{r.low_id}']
    else:blocks.append('No hay pareja que cumpla los umbrales predefinidos; no se relajaron tras ver los resultados.')
    blocks+=['### Evaluación humana propuesta, no realizada',
        'Consultas fijadas antes de escuchar, sugerencias original/PCA con etiquetas ocultas, orden alternado y fragmentos de igual duración elegidos de forma consistente. Voluntarios con consentimiento e identificadores anónimos puntúan similitud y utilidad de 1 a 5. Reportar participantes, consultas, distribuciones y comentarios negativos; una muestra pequeña no generaliza a todos los oyentes. No hay puntuaciones humanas inventadas.',
        '## 10. Qué mejoró y qué no puede afirmarse',
        'Se añadió selección separada por artistas, comparación acotada de preparación, tamaños mínimos, búsqueda hasta 24 grupos, regla común orientada a segmentación, prueba adicional sin artistas previos y benchmarks repetidos. No se cambió la métrica para demostrar éxito: se declaró el cambio respecto al objetivo BIC anterior y se mantienen sus diagnósticos.',
        'Una nueva Silhouette no es directamente comparable con la anterior cuando cambia la población, el ajuste o la geometría. La comparación controlada entre StandardScaler y Yeo-Johnson es la tabla de validación con los mismos datos y referencia. El test reservado es una comprobación posterior, no otra oportunidad de ajuste. Si no mejora, debe reportarse.',
        '## 11. Reproducibilidad y asistencia de IA',
        'Ejecutar Proyecto_Mejorado.ipynb completo o python mejorar_proyecto.py desde esta carpeta. Semillas, criterios, búsqueda completa, IDs de las particiones, huellas y versiones acompañan la entrega. No hace falta descargar audio, abrir una API o tener internet para repetir con el CSV incluido. Se utilizó ChatGPT/Codex para código, documentación y análisis; los estudiantes deben revisar y declarar la asistencia. La preparación no garantiza una calificación.',
        '## Referencias',
        'Pandya, M. (s. f.). Spotify Tracks Dataset [Conjunto de datos]. Hugging Face. https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset',
        'Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html',
        'Scikit-learn developers. (s. f.). PowerTransformer y GroupShuffleSplit [Documentación]. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html ; https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html']
    # Comparación controlada sobre las mismas canciones y geometría de referencia.
    old=pd.read_csv(ROOT/'data/referencia_anterior/comparativa.csv').query("split=='train' and evaluation_space=='original'")
    revised=common.query("split=='train'")
    history=old[['model','space','groups','silhouette']].merge(revised[['model','space','groups','silhouette']],on=['model','space'],suffixes=('_anterior','_revision'))
    history['delta_silhouette']=history.silhouette_revision-history.silhouette_anterior
    history.to_csv(OUT/'comparacion_controlada_anterior.csv',index=False)
    position=blocks.index('## 10. Qué mejoró y qué no puede afirmarse')+1
    gain=frozen['mean_validation_silhouette']['yeo_johnson']-frozen['mean_validation_silhouette']['standard']
    blocks[position:position]=[table(history),
        'Esta comparación usa las mismas 8.000 canciones, las mismas diez variables y StandardScaler de referencia. La revisión modifica preparación y selección: no permite atribuir toda la diferencia solo a Yeo-Johnson. Se observa mejora geométrica en GMM y empeoramiento en K-Means; no es una mejora universal, ni demuestra mejor ajuste de densidad o utilidad para oyentes.',
        f'La ventaja media de Yeo-Johnson en validación fue {gain:+.5f}, frente al umbral predefinido de 0,005. Es pequeña y está cerca del umbral; no se evaluó significación estadística ni robustez a múltiples particiones de validación. No debe presentarse como una superioridad contundente.']
    if len(examples):
        r=examples.iloc[0]
        comparison_html=table(pd.read_csv(OUT/'comparacion_escucha.csv'))
        cards=''.join('<article><p>'+label+'</p><h2>'+html.escape(str(r[prefix+'_song']))+'</h2><p>'+html.escape(str(r[prefix+'_artist']))+'</p><strong>Popularidad histórica '+str(int(r[prefix+'_popularity']))+'/100</strong><p><a href="https://open.spotify.com/track/'+html.escape(str(r[prefix+'_id']))+'" target="_blank" rel="noopener noreferrer">Abrir en Spotify</a></p></article>' for label,prefix in [('Canción A','high'),('Canción B','low')])
        document='<!doctype html><html lang="es"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Cierre musical</title><style>body{max-width:1000px;margin:40px auto;padding:24px;background:#0c1524;color:#edf4ff;font:18px/1.6 system-ui}section{display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:24px}article{background:#17263b;border:1px solid #38516c;border-radius:16px;padding:24px}a{color:#9ff0c8}strong{font-size:24px}.note{color:#b7c8db;font-size:15px}</style><h1>Sonidos cercanos. Popularidades distintas.</h1><section>'+cards+'</section><p>Distancia en las diez características estandarizadas: '+f'{r.distance:.3f}'+'</p><p class="note">Ejemplo elegido por la menor distancia entre '+str(len(examples))+' consultas elegibles: popularidad ≥70 frente a ≤30, mismo perfil. Es una ilustración deliberada, no una prueba representativa ni una garantía de similitud perceptiva.</p><p>Escucha fragmentos de igual duración y contrasta lo que percibes con los datos. Si suenan diferentes, identifica qué no capturan estas características.</p><p class="note">No se incluye audio ni se analizan versos, estribillos o melodía. Los enlaces requieren conexión.</p><p><a href="informe.html">Volver al informe completo</a></p></html>'
        (OUT/'cierre_escucha.html').write_text(document,'utf-8')
    (OUT/'informe.md').write_text('\n\n'.join(blocks),'utf-8');(OUT/'informe.html').write_text(render_report(blocks),'utf-8')
    return key


In [22]:
def run():
    OUT.mkdir(exist_ok=True);(OUT/'figuras').mkdir(exist_ok=True)
    (OUT/'protocolo_ejecutado.json').write_text((ROOT/'protocolo_mejora.json').read_text('utf-8'),'utf-8')
    start=time.perf_counter()
    train,fit,val,test,audit=splits();print('Particiones verificadas: '+str({k:audit[k] for k in ['development','inner_fit','validation','holdout','eligible_holdout']}),flush=True)
    frozen,searches,selected=search(fit,val)
    print('Selección congelada: '+frozen['preprocessing'],flush=True)
    comparison,stable,profiles,pops,loads,labels,common_train,common_test,summary=final_experiment(train,test,frozen)
    config=pd.DataFrame(frozen['selected']).sort_values('val_sil_common',ascending=False).iloc[0];key=config.model+'_'+config.space
    examples=listening_example(train,test,labels,common_train,common_test,key)
    report(audit,frozen,searches,selected,comparison,stable,profiles,pops,loads,summary,examples)
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','matplotlib','nbformat','threadpoolctl']};versions['python']=platform.python_version();dump('versiones.json',versions)
    assert len(comparison)==16 and len(stable)==48
    assert np.isfinite(comparison[['silhouette','davies_bouldin','calinski_harabasz']]).all().all()
    assert len(train)>=500 and len(FEATURES)>=8 and summary['pca_variance']>=.9
    text=(OUT/'informe.md').read_text('utf-8');assert re.findall(r'^### (\d+)\. ',text,re.M)==[str(i) for i in range(1,13)]
    assert not frozen['selection_uses_holdout'] and not frozen['selection_uses_popularity']
    dump('validacion.json',dict(status='passed',comparison_rows=len(comparison),stability_pairs=len(stable),questions=12,shared_artists_fit_validation=0,shared_artists_previous_holdout=0,holdout_ids_not_previously_used=True,protocol_sha256=frozen['protocol_sha256'],elapsed_seconds=time.perf_counter()-start))
    print(comparison.query("split=='test' and evaluation_space=='reference_original'")[[ 'model','space','groups','silhouette','davies_bouldin','calinski_harabasz']].to_string(index=False),flush=True)
    print('FINALIZADO: resultados/informe.html',flush=True)
    # La pregunta principal se responde después de congelar los modelos.
    from analizar_popularidad import ejecutar_objetivo
    ejecutar_objetivo()
    from analizar_contextos import ejecutar_ampliacion
    ejecutar_ampliacion()



## Objetivo principal: perfiles y alta popularidad

In [23]:
"""Contraste posterior al clustering: popularidad, perfiles y rasgos descriptivos."""
from pathlib import Path
import hashlib
import json
import re
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits
import mejorar_proyecto as core

POP_ROOT = Path.cwd()
POP_OUT = POP_ROOT/'resultados'


In [24]:
def pop_artists_bootstrap(frame, labels, leader, threshold, repetitions, seed):
    valid=frame.popularity.notna().to_numpy()
    f=frame.loc[valid].reset_index(drop=True);labs=np.asarray(labels)[valid]
    _,groups=np.unique(core.artist_groups(f),return_inverse=True)
    n_groups=int(groups.max()+1)
    in_profile=(labs==leader);high=(f.popularity.to_numpy()>=threshold)
    # Cada componente de artistas se remuestrea como una unidad completa.
    counts=np.zeros((n_groups,4),dtype=float)
    np.add.at(counts[:,0],groups,in_profile.astype(float))
    np.add.at(counts[:,1],groups,(in_profile&high).astype(float))
    np.add.at(counts[:,2],groups,(~in_profile).astype(float))
    np.add.at(counts[:,3],groups,((~in_profile)&high).astype(float))
    total=counts.sum(axis=0)
    if total[0]==0 or total[2]==0:return dict(diff_low=np.nan,diff_high=np.nan,lift_low=np.nan,lift_high=np.nan,artist_components=n_groups)
    rng=np.random.default_rng(seed);diff=[];lift=[]
    for start in range(0,repetitions,200):
        ix=rng.integers(0,n_groups,size=(min(200,repetitions-start),n_groups))
        draw=counts[ix].sum(axis=1)
        good=(draw[:,0]>0)&(draw[:,2]>0)
        draw=draw[good]
        r1=draw[:,1]/draw[:,0];r0=draw[:,3]/draw[:,2]
        base=(draw[:,1]+draw[:,3])/(draw[:,0]+draw[:,2])
        diff.extend(r1-r0);positive=base>0;lift.extend(r1[positive]/base[positive])
    if not diff:return dict(diff_low=np.nan,diff_high=np.nan,lift_low=np.nan,lift_high=np.nan,artist_components=n_groups)
    a,b=np.quantile(diff,[.025,.975]);c,d=np.quantile(lift,[.025,.975]) if lift else (np.nan,np.nan)
    return dict(diff_low=float(a),diff_high=float(b),lift_low=float(c),lift_high=float(d),artist_components=n_groups)


In [25]:
def pop_profile_summary(frame, labels, name, split, threshold):
    rows=[];valid=frame.popularity.notna();global_rate=float((frame.loc[valid,'popularity']>=threshold).mean())
    for cluster in sorted(np.unique(labels)):
        f=frame.loc[(np.asarray(labels)==cluster)&valid.to_numpy()];n=len(f);high=int((f.popularity>=threshold).sum());rate=high/n if n else np.nan
        rows.append(dict(configuration=name,split=split,cluster=int(cluster),threshold=threshold,n=n,high_count=high,high_rate=rate,global_rate=global_rate,lift=rate/global_rate if global_rate else np.nan,median=float(f.popularity.median()),q25=float(f.popularity.quantile(.25)),q75=float(f.popularity.quantile(.75))))
    return pd.DataFrame(rows)


In [26]:
def pop_reproduce(train,previous_test,frozen):
    prep=core.pipeline(frozen['preprocessing']).fit(train[core.FEATURES]);X=prep.transform(train[core.FEATURES])
    pca=PCA(n_components=.9,svd_solver='full').fit(X)
    reference=core.pipeline('standard').fit(train[core.FEATURES])
    models={};check=[]
    for cfg in frozen['selected']:
        key=cfg['model']+'_'+cfg['space'];A=X if cfg['space']=='original' else pca.transform(X)
        est=core.estimator(cfg['model'],cfg['k'],cfg['covariance']);_,ok,_=core.fit_model(est,A)
        if not ok:raise RuntimeError('No converge reproducción '+key)
        known=pd.read_csv(POP_OUT/f'asignaciones_{key}.csv').set_index('track_id')
        predicted=est.predict(A);ari=adjusted_rand_score(known.loc[train.track_id,'cluster'],predicted)
        B=prep.transform(previous_test[core.FEATURES]);B=B if cfg['space']=='original' else pca.transform(B)
        ari_test=adjusted_rand_score(known.loc[previous_test.track_id,'cluster'],est.predict(B))
        if ari<.999999 or ari_test<.999999:raise RuntimeError('No reproduce particiones '+key)
        models[key]=(est,cfg['space'],predicted);check.append(dict(configuration=key,ari_train=float(ari),ari_previous_test=float(ari_test)))
    return prep,pca,reference,models,check


In [27]:
def pop_new_evaluation(protocol,previous):
    raw=core.clean_data(pd.read_csv(POP_ROOT/'data/raw/spotify_tracks.csv'))
    used_artists=core.artist_union(previous)
    eligible=(~raw.track_id.isin(previous.track_id)) & raw.artists.map(lambda v:bool(core.artists(v)) and not bool(core.artists(v)&used_artists))
    eligible &= ~np.isin(core.signatures(raw),core.signatures(previous))
    pool=raw.loc[eligible]
    if len(pool)<protocol['nueva_evaluacion']:raise ValueError('No hay suficientes canciones reservables')
    test=pool.sample(n=protocol['nueva_evaluacion'],random_state=protocol['semilla_nueva_evaluacion']).reset_index(drop=True)
    assert not core.artist_union(previous)&core.artist_union(test)
    assert not set(previous.track_id)&set(test.track_id)
    assert not set(core.signatures(previous))&set(core.signatures(test))
    test.to_csv(POP_OUT/'prueba_objetivo_popularidad.csv',index=False)
    return test,len(pool)


In [28]:
def pop_characteristics(train,labels,leader,name):
    a=train.loc[np.asarray(labels)==leader,core.FEATURES];b=train.loc[np.asarray(labels)!=leader,core.FEATURES]
    med_a=a.median();med_b=b.median();sd=train[core.FEATURES].std()
    result=pd.DataFrame(dict(configuration=name,cluster=leader,variable=core.FEATURES,profile_median=med_a.to_numpy(),rest_median=med_b.to_numpy(),difference=(med_a-med_b).to_numpy(),standardized_difference=((med_a-med_b)/sd).to_numpy()))
    return result.sort_values('standardized_difference',key=abs,ascending=False)


In [29]:
def pop_write_report(protocol,primary,leaders,summary,effects,features,reproduction,new_test,pool_size):
    base=(POP_OUT/'informe.md').read_text('utf-8')
    # La ejecución independiente reconstruye el método desde resultados verificados.
    # No necesita conservar otro informe con un objetivo diferente.
    if '## 2. Datos y preparación sin fuga' not in base:
        core.report(json.loads((POP_OUT/'auditoria.json').read_text('utf-8')),
            json.loads((POP_OUT/'seleccion_congelada.json').read_text('utf-8')),
            pd.read_csv(POP_OUT/'busqueda_completa.csv'),pd.read_csv(POP_OUT/'seleccion_por_preparacion.csv'),
            pd.read_csv(POP_OUT/'comparativa.csv'),pd.read_csv(POP_OUT/'estabilidad.csv'),
            pd.read_csv(POP_OUT/'perfiles.csv'),pd.read_csv(POP_OUT/'popularidad.csv'),
            pd.read_csv(POP_OUT/'cargas_pca.csv',index_col=0),
            json.loads((POP_OUT/'transformacion.json').read_text('utf-8')),
            pd.read_csv(POP_OUT/'ejemplos_escucha.csv'))
        base=(POP_OUT/'informe.md').read_text('utf-8')
    method_start=base.index('## 2. Datos y preparación sin fuga')
    method_end=base.index('## 9. Cierre con canciones reales')
    method=base[method_start:method_end]
    for number,title in [(2,'Datos y preparación sin fuga'),(3,'Protocolo y selección'),(4,'PCA e interpretación'),(5,'Tabla comparativa obligatoria y comparación justa'),(6,'Estabilidad y costo'),(7,'Interpretación y acciones por perfil')]:
        method=method.replace(f'## {number}. {title}',f'### Método: {title}')
    method=method.replace('## 8. Las doce respuestas de la rúbrica','## 8. Las doce respuestas de la rúbrica')
    method=method.replace('### Popularidad como análisis secundario','### Distribuciones de popularidad después del agrupamiento')
    method=method.replace('La decisión práctica es segmentar el catálogo','La segmentación permite contrastar perfiles y popularidad')
    method=method.replace('La popularidad es un análisis descriptivo secundario.','La popularidad se contrasta después de congelar los modelos.')
    e=effects.query("configuration==@primary and split=='nueva_evaluacion' and threshold==70").iloc[0]
    leader=leaders[primary];train_row=summary.query("configuration==@primary and split=='entrenamiento' and threshold==70 and cluster==@leader").iloc[0]
    test_row=summary.query("configuration==@primary and split=='nueva_evaluacion' and threshold==70 and cluster==@leader").iloc[0]
    if e.diff_low>0:verdict='La nueva evaluación aporta evidencia de una mayor concentración de alta popularidad en el perfil candidato, dentro de esta fuente y estos criterios.'
    elif e.diff_high<0:verdict='La asociación se invierte en la nueva evaluación: el perfil candidato no conserva la ventaja observada en entrenamiento.'
    else:verdict='No se confirma una mayor concentración en la nueva evaluación: el intervalo de la diferencia incluye cero.'
    method=method.replace('### 12. Recomendación final','### 12. Recomendación final\n\n'+f'Para la pregunta de popularidad, el contraste principal de {primary} da lift {test_row.lift:.2f} e IC95 de diferencia [{100*e.diff_low:+.2f}, {100*e.diff_high:+.2f}] pp. {verdict} No se recomienda convertir el perfil en regla comercial ni escoger otra configuración porque produzca una asociación más favorable. La recomendación geométrica que sigue se refiere a la calidad de agrupamiento, no a predicción de éxito.')
    feature_primary=features.query('configuration==@primary')
    dominant='; '.join(f'{r.variable}: mediana {r.profile_median:.3f} frente a {r.rest_median:.3f} en el resto' for r in feature_primary.head(4).itertuples())
    blocks=['# ¿Existen perfiles sonoros que concentren canciones de alta popularidad y qué características los distinguen?',
        'Proyecto de Bloque 3 · Aprendizaje no supervisado · K-Means y GMM antes y después de PCA. Completar autores y fecha de entrega.',
        '## Resumen del resultado',
        f'El contraste principal usa {primary}, seleccionado antes por calidad geométrica y sin popularidad. El perfil candidato es el grupo {leader}, elegido en entrenamiento por su proporción de canciones con popularidad ≥70: {int(train_row.high_count)}/{int(train_row.n)} ({train_row.high_rate:.2%}). En 2.000 canciones adicionales de artistas no utilizados antes, el perfil presenta {int(test_row.high_count)}/{int(test_row.n)} ({test_row.high_rate:.2%}), frente a una tasa global de {test_row.global_rate:.2%}. Su lift es {test_row.lift:.2f}. {verdict}',
        f'Diferencia frente al resto: {100*e.rate_difference:+.2f} puntos porcentuales, IC95 por artistas [{100*e.diff_low:+.2f}, {100*e.diff_high:+.2f}]. Las características que más distinguen el perfil en entrenamiento son: {dominant}. Son rasgos descriptivos del grupo, no causas del éxito.',
        '## 1. Objetivo y alcance',protocol['objetivo'],
        'Objetivo general: descubrir perfiles utilizando exclusivamente características sonoras, determinar si alguno concentra canciones de alta popularidad y describir los atributos que lo distinguen. Objetivos específicos: limpiar y preparar datos; comparar K-Means y GMM antes/después de PCA; identificar candidatos en entrenamiento; evaluar concentración en canciones adicionales; comprobar sensibilidad y comunicar límites.',
        'El clustering es el método para responder esta pregunta, no el fin comercial del proyecto. No se convierte el trabajo en un predictor supervisado ni se usa popularidad para escoger algoritmos, preparación, K o covarianza.',
        '## 2. Cómo se define y mide alta popularidad',
        'Definición operativa principal: popularity ≥70 en la escala histórica 0–100 del dataset. No equivale a ventas, ingresos o éxito universal. Los umbrales 60 y 80 son sensibilidad fijada antes de esta evaluación; no se elige el que produzca la mejor historia. Ceros se conservan como valores del archivo, sin interpretarlos como ausencia real de oyentes.',
        'Concentración = canciones ≥70 / canciones con popularidad válida del perfil. Lift = tasa del perfil / tasa global del mismo conjunto: 1 indica igual concentración; 2 significa el doble de la tasa observada, no el doble de probabilidad causal de éxito. La diferencia principal compara perfil y resto, expresada en puntos porcentuales.',
        'Protocolo: algoritmos y parámetros ya estaban congelados por separación geométrica. Para cada una de las cuatro configuraciones se elige el perfil candidato exclusivamente por la mayor tasa ≥70 de entrenamiento, con desempate por ID. El modelo principal se fija por el mayor Silhouette de validación previamente registrado. Se preserva esa elección en sensibilidad, sin seleccionar nuevos perfiles con los resultados de prueba.',
        '## 3. Comprobación en canciones no utilizadas antes',
        f'Se reservaron {len(new_test)} canciones con semilla {protocol["semilla_nueva_evaluacion"]}, de {pool_size} elegibles. Se excluyeron los 10.000 IDs históricos, las 2.000 canciones de la evaluación adicional anterior, todos sus artistas individuales y los vectores acústicos iguales a siete decimales. No se eligieron canciones por su popularidad. La fuente histórica es la misma: no se presenta como evidencia temporal o externa.',
        'La evaluación anterior de 2.000 canciones ya había sido observada y se muestra como retrospectiva. La conclusión principal utiliza la nueva evaluación. La decisión de revisar el objetivo está informada por análisis previos: este protocolo documenta criterios anteriores a este nuevo cálculo, pero no es un prerregistro externo.',
        '![Concentración de alta popularidad por perfil](figuras/concentracion_popularidad.png)',
        core.table(summary.query('threshold==70')),
        '### Contraste principal y comparación de configuraciones',core.table(effects.query("split=='nueva_evaluacion' and threshold==70")),
        'Intervalos percentiles de 2.000 remuestreos por componentes de artistas colaboradores; cada unidad se conserva completa. Son condicionales a los modelos y perfiles elegidos y no incluyen toda la incertidumbre de volver a entrenar. Las cuatro configuraciones comparten canciones y no son cuatro confirmaciones independientes; solo la configuración principal define el contraste primario. Los análisis restantes son exploratorios.',
        '![Lift y su incertidumbre en la evaluación nueva](figuras/lift_popularidad.png)',
        '## 4. Qué características distinguen los perfiles candidatos',
        'Las tablas usan medianas en unidades originales y contraste perfil−resto. La diferencia estandarizada solo ordena atributos descriptivos; no es un coeficiente causal ni una prueba de importancia predictiva. El perfil se define por todas las variables, no por una sola regla musical.',core.table(feature_primary),
        '![Características del perfil candidato principal](figuras/rasgos_popularidad.png)',
        '### Perfiles candidatos de las cuatro configuraciones',core.table(features),
        '## 5. Sensibilidad y distribución completa',core.table(effects.query("split=='nueva_evaluacion'")),
        'La sensibilidad mantiene el perfil candidato identificado con umbral 70. Revisar si el signo de la diferencia se conserva con 60 y 80 y cuán amplios son sus intervalos; no elegir un umbral después de ver los resultados. Con pocos casos ≥80 la incertidumbre puede ser grande.',
        '![Popularidad completa por perfil](figuras/distribucion_popularidad.png)',
        'Las cajas muestran mediana y dispersión: una proporción alta puede coexistir con muchas canciones de baja popularidad. La superposición impide convertir un perfil en garantía individual.',
        '## 6. Respuesta y límites',verdict,
        f'La respuesta debe citar el contraste de {primary}, perfil {leader}: tasa {test_row.high_rate:.2%}, lift {test_row.lift:.2f}, diferencia {100*e.rate_difference:+.2f} pp e intervalo [{100*e.diff_low:+.2f}, {100*e.diff_high:+.2f}]. Si no se confirma, no se afirma que no exista ningún patrón posible: solo que este candidato y este diseño no lo confirman.',
        'El análisis no distingue causas de popularidad: promoción, artista, idioma, exposición en playlists y momento del lanzamiento pueden explicar asociaciones. Los atributos agregados no describen melodía, letra o estructura verso/estribillo. No se recomienda una receta de producción ni se afirma un porcentaje de precisión de éxitos.',
        '## 7. Experimento obligatorio de clustering y PCA']
    blocks.extend(method.strip().split('\n\n'))
    blocks+=['## 9. Cierre de la presentación',
        'Primero presentar la tasa global, la del perfil candidato y sus intervalos. Después mostrar sus rasgos acústicos y escuchar ejemplos que permitan contrastarlos. El archivo cierre_escucha.html conserva una pareja ilustrativa de canciones cercanas con popularidades distintas: sirve para explicar los límites de una asociación, no para reemplazar el contraste principal. No se incluye audio ni una supuesta estructura ganadora.',
        '## 10. Reproducibilidad, declaración de IA y referencias',
        'Ejecutar Proyecto_Mejorado.ipynb completo o python mejorar_proyecto.py. El núcleo ajusta los cuatro modelos y este módulo reproduce sus particiones con ARI=1 antes de estudiar popularidad. Protocolo, selección de perfiles, IDs y conteos, remuestreos, tablas y figuras acompañan la entrega. Se utilizó ChatGPT/Codex para implementación y documentación; revisar y declarar esa asistencia. No hay evaluaciones humanas inventadas ni garantía de nota.',
        'Pandya, M. (s. f.). Spotify Tracks Dataset [Conjunto de datos]. Hugging Face. https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset',
        'Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html',
        'Scikit-learn developers. (s. f.). PowerTransformer y GroupShuffleSplit [Documentación]. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html ; https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html']
    document=core.render_report(blocks).replace('Segmentación musical — Informe mejorado','Perfiles sonoros y alta popularidad')
    (POP_OUT/'informe.md').write_text('\n\n'.join(blocks),'utf-8');(POP_OUT/'informe.html').write_text(document,'utf-8')
    return dict(primary_model=primary,primary_profile=leader,verdict=verdict,profile_rate=float(test_row.high_rate),baseline_rate=float(test_row.global_rate),lift=float(test_row.lift),difference_pp=float(100*e.rate_difference),difference_ci_pp=[float(100*e.diff_low),float(100*e.diff_high)])


In [30]:
def ejecutar_objetivo():
    protocol=json.loads((POP_ROOT/'protocolo_popularidad.json').read_text('utf-8'))
    frozen=json.loads((POP_OUT/'seleccion_congelada.json').read_text('utf-8'))
    train=core.clean_data(pd.read_csv(POP_OUT/'entrenamiento.csv'))
    previous_test=core.clean_data(pd.read_csv(POP_OUT/'prueba_reservada.csv'))
    prep,pca,reference,models,reproduction=pop_reproduce(train,previous_test,frozen)
    primary_row=pd.DataFrame(frozen['selected']).sort_values(['val_sil_common','model','space'],ascending=[False,True,True]).iloc[0]
    primary=primary_row.model+'_'+primary_row.space
    leaders={};train_stats=[]
    for key,(_,_,labels) in models.items():
        stats=pop_profile_summary(train,labels,key,'entrenamiento',70);train_stats.append(stats)
        leaders[key]=int(stats.sort_values(['high_rate','cluster'],ascending=[False,True]).iloc[0].cluster)
    # Archivo escrito antes de cargar las nuevas observaciones de evaluación.
    decision=dict(objective=protocol['objetivo'],threshold=70,primary_model=primary,leaders=leaders,models_reselected_by_popularity=False,profile_selection_uses_training_only=True,protocol_sha256=hashlib.sha256((POP_ROOT/'protocolo_popularidad.json').read_bytes()).hexdigest())
    core.dump('perfiles_candidatos_congelados.json',decision)
    old=pd.concat([pd.read_csv(POP_ROOT/'data/referencia_anterior/train.csv'),pd.read_csv(POP_ROOT/'data/referencia_anterior/test.csv'),previous_test],ignore_index=True)
    test,pool_size=pop_new_evaluation(protocol,core.clean_data(old))
    all_stats=list(train_stats);effects=[];features=[];new_labels={}
    for key,(est,space,lt) in models.items():
        leader=leaders[key];features.append(pop_characteristics(train,lt,leader,key))
        for split,frame in [('evaluacion_anterior_observada',previous_test),('nueva_evaluacion',test)]:
            A=prep.transform(frame[core.FEATURES]);A=A if space=='original' else pca.transform(A);labs=est.predict(A)
            if split=='nueva_evaluacion':new_labels[key]=labs
            frame.assign(cluster=labs).to_csv(POP_OUT/f'popularidad_asignaciones_{split}_{key}.csv',index=False)
            for threshold in [70]+protocol['sensibilidad']:
                stats=pop_profile_summary(frame,labs,key,split,threshold);all_stats.append(stats)
                row=stats.loc[stats.cluster==leader].iloc[0];rest=frame.loc[(labs!=leader)&frame.popularity.notna().to_numpy(),'popularity'];rest_rate=float((rest>=threshold).mean())
                ci=pop_artists_bootstrap(frame,labs,leader,threshold,protocol['bootstrap'],protocol['semilla_bootstrap'])
                effects.append(dict(configuration=key,split=split,cluster=leader,threshold=threshold,n_profile=int(row.n),high_count=int(row.high_count),profile_rate=float(row.high_rate),rest_rate=rest_rate,global_rate=float(row.global_rate),lift=float(row.lift),rate_difference=float(row.high_rate-rest_rate),**ci))
    summary=pd.concat(all_stats,ignore_index=True);effects=pd.DataFrame(effects);features=pd.concat(features,ignore_index=True)
    summary.to_csv(POP_OUT/'concentracion_por_perfil.csv',index=False);effects.to_csv(POP_OUT/'contrastes_popularidad.csv',index=False);features.to_csv(POP_OUT/'rasgos_perfiles_populares.csv',index=False)
    principal=summary.query("configuration==@primary and split=='nueva_evaluacion' and threshold==70")
    plt.figure(figsize=(7,4));plt.bar(principal.cluster.astype(str),100*principal.high_rate,color='#358ca1');plt.axhline(100*principal.global_rate.iloc[0],ls='--',color='black',label='Tasa global');plt.xlabel('Perfil sonoro');plt.ylabel('Canciones con popularidad ≥70 (%)');plt.title('Nueva evaluación: '+primary);plt.legend();core.figure('concentracion_popularidad')
    eff=effects.query("split=='nueva_evaluacion' and threshold==70").reset_index(drop=True)
    plt.figure(figsize=(8,4))
    for i,r in eff.iterrows():plt.plot([r.lift_low,r.lift_high],[i,i],color='#358ca1');plt.scatter(r.lift,i,color='#123e50')
    plt.axvline(1,ls='--',color='black');plt.yticks(range(len(eff)),eff.configuration);plt.xlabel('Lift respecto a tasa global (IC95 por artistas)');plt.title('Concentración del perfil candidato, fijado en entrenamiento');core.figure('lift_popularidad')
    fp=features.query('configuration==@primary').sort_values('standardized_difference');plt.figure(figsize=(8,4));plt.barh(fp.variable,fp.standardized_difference,color='#358ca1');plt.axvline(0,color='black');plt.xlabel('Diferencia de medianas perfil−resto / DE de entrenamiento');plt.title('Rasgos descriptivos del perfil candidato: '+primary);core.figure('rasgos_popularidad')
    fig,axs=plt.subplots(2,2,figsize=(10,7))
    for ax,(key,labs) in zip(axs.flat,new_labels.items()):
        groups=sorted(np.unique(labs));ax.boxplot([test.loc[labs==g,'popularity'].dropna() for g in groups],tick_labels=[str(g) for g in groups],showfliers=True,flierprops=dict(marker='.',markersize=3,alpha=.35));ax.set(title=key,xlabel='Perfil',ylabel='Popularidad histórica');ax.axhline(70,color='red',ls='--',lw=.8)
    core.figure('distribucion_popularidad')
    conclusion=pop_write_report(protocol,primary,leaders,summary,effects,features,reproduction,test,pool_size)
    text=(POP_OUT/'informe.md').read_text('utf-8');assert re.findall(r'^### (\d+)\. ',text,re.M)==[str(i) for i in range(1,13)]
    assert len(effects)==24 and len(summary.query("split=='entrenamiento'"))>=8
    core.dump('resultado_objetivo.json',dict(**conclusion,n_new_evaluation=len(test),eligible_pool=pool_size,bootstrap_repetitions=protocol['bootstrap'],reproduction=reproduction,status='passed',exploratory_previous_evaluation=True,models_reselected_by_popularity=False))
    print(json.dumps(conclusion,ensure_ascii=False,indent=2),flush=True)
    return conclusion



## Ejecutar todo el experimento
Primero se congelan los modelos sin popularidad. Después se reproduce cada partición y se evalúa el objetivo. No se eligen parámetros mirando la nueva prueba.

## Ampliación: contextos y perfiles más detallados
Se explora K=3–10 y se analiza género únicamente después del ajuste. La nueva prueba utiliza hasta 6.000 canciones sin artistas compartidos con las etapas anteriores. Contextos secundarios con corrección múltiple; ninguna regla se elige por resultados de prueba.

In [31]:
"""Ampliación controlada: mayor granularidad y contextos de género posteriores al ajuste."""
from pathlib import Path
import json
import hashlib
import re
import html
import itertools
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits
import mejorar_proyecto as core
import analizar_popularidad as pop

CTX_ROOT=Path.cwd()
CTX_OUT=CTX_ROOT/'resultados/ampliacion'
CTX_P=json.loads((CTX_ROOT/'protocolo_contextos.json').read_text('utf-8'))


In [32]:
def ctx_save(name,value):
    (CTX_OUT/name).write_text(json.dumps(value,ensure_ascii=False,indent=2,default=lambda x:x.item() if hasattr(x,'item') else str(x)),'utf-8')


In [33]:
def ctx_figure(name):
    plt.tight_layout();plt.savefig(CTX_OUT/'figuras'/f'{name}.png',dpi=140,bbox_inches='tight');plt.close()


In [34]:
def ctx_tags(raw,frames):
    tags=raw.groupby('track_id',sort=True).track_genre.agg(lambda values:set(str(v).strip().casefold() for v in values if pd.notna(v))).to_dict()
    result=[]
    for frame in frames:
        result.append({name:np.array([bool(tags.get(t,set())&set(genres)) for t in frame.track_id]) for name,genres in CTX_P['contexts'].items()})
    return result


In [35]:
def ctx_search(fit,val,preprocessing):
    prep=core.pipeline(preprocessing).fit(fit[core.FEATURES]);A=prep.transform(fit[core.FEATURES]);B=prep.transform(val[core.FEATURES])
    ref=core.pipeline('standard').fit(fit[core.FEATURES]);V=ref.transform(val[core.FEATURES])
    pca=PCA(n_components=.9,svd_solver='full').fit(A)
    rows=[];choices=[]
    for family in ['KMeans','GMM']:
        for space,X,T in [('original',A,B),('pca',pca.transform(A),pca.transform(B))]:
            block=[]
            for k in CTX_P['k_grid']:
                for cov in (['NA'] if family=='KMeans' else CTX_P['covariances']):
                    est=core.estimator(family,k,cov);seconds,ok,nwarn=core.fit_model(est,X);lt=est.predict(X);lv=est.predict(T)
                    min_train=float(np.bincount(lt,minlength=k).min()/len(lt));min_val=float(np.bincount(lv,minlength=k).min()/len(lv))
                    sil=core.silhouette(V,lv)
                    row=dict(model=family,space=space,k=k,covariance=cov,converged=ok,warnings=nwarn,min_train_fraction=min_train,min_val_fraction=min_val,val_sil_common=sil,train_sil_native=core.silhouette(X,lt),bic=float(est.bic(X)) if family=='GMM' else np.nan,inertia=float(est.inertia_) if family=='KMeans' else np.nan,fit_seconds=seconds)
                    row['eligible']=bool(ok and np.isfinite(sil) and min_train>=CTX_P['min_cluster_fraction'] and min_val>=CTX_P['min_cluster_fraction']);rows.append(row);block.append(row)
            d=pd.DataFrame(block);eligible=d.loc[d.eligible]
            if eligible.empty:raise RuntimeError('Ningún modelo fino elegible: '+family+'/'+space)
            near=eligible.loc[eligible.val_sil_common>=eligible.val_sil_common.max()-CTX_P['silhouette_tolerance']]
            best=near.sort_values(['k','bic'],na_position='last').iloc[0];choices.append(best.to_dict())
            fig,axs=plt.subplots(1,2,figsize=(10,3.7))
            for cov,g in d.groupby('covariance'):
                axs[0].plot(g.k,g.val_sil_common,'o-',label=cov);axs[1].plot(g.k,g.bic if family=='GMM' else g.inertia,'o-',label=cov)
            axs[0].set(xlabel='K',ylabel='Silhouette validación común',title=family+'/'+space);axs[1].set(xlabel='K',ylabel='BIC' if family=='GMM' else 'Inercia');axs[0].legend();axs[1].legend();ctx_figure('busqueda_'+family+'_'+space)
            print(f'Fino {family}/{space}: K={int(best.k)}; validación={best.val_sil_common:.4f}',flush=True)
    pd.DataFrame(rows).to_csv(CTX_OUT/'busqueda.csv',index=False);pd.DataFrame(choices).to_csv(CTX_OUT/'seleccion.csv',index=False)
    return choices


In [36]:
def ctx_leader(frame,labels,minimum):
    rows=[]
    for label in sorted(np.unique(labels)):
        f=frame.loc[np.asarray(labels)==label];valid=f.popularity.dropna()
        if len(valid)>=minimum and len(frame)-len(f)>=minimum:rows.append((float((valid>=70).mean()),int(label)))
    if not rows:return None
    return sorted(rows,key=lambda x:(-x[0],x[1]))[0][1]


In [37]:
def ctx_effect(frame,labels,leader,alpha=.05,min_side=1):
    valid=frame.popularity.notna().to_numpy();f=frame.loc[valid].reset_index(drop=True);labs=np.asarray(labels)[valid]
    inside=labs==leader;high=f.popularity.to_numpy()>=70;n=int(inside.sum());rest=int((~inside).sum());hits=int((inside&high).sum());total_high=int(high.sum())
    rate=hits/n if n else np.nan;other=float(high[~inside].mean()) if rest else np.nan;base=float(high.mean()) if len(f) else np.nan
    result=dict(n=len(f),n_profile=n,high_count=hits,total_high=total_high,profile_rate=rate,rest_rate=other,global_rate=base,lift=rate/base if base else np.nan,coverage=n/len(f) if len(f) else np.nan,capture=hits/total_high if total_high else np.nan,difference=rate-other,diff_low=np.nan,diff_high=np.nan,lift_low=np.nan,lift_high=np.nan,alpha=alpha,status='ok')
    if min(n,rest)<min_side:result['status']='muestra insuficiente';return result
    if not total_high:result['status']='sin casos de alta popularidad';return result
    _,groups=np.unique(core.artist_groups(f),return_inverse=True);ng=int(groups.max()+1);counts=np.zeros((ng,4))
    for col,values in enumerate([inside,inside&high,~inside,(~inside)&high]):np.add.at(counts[:,col],groups,values.astype(float))
    rng=np.random.default_rng(CTX_P['bootstrap_seed']);differences=[];lifts=[]
    for start in range(0,CTX_P['bootstrap'],100):
        draw=counts[rng.integers(0,ng,size=(min(100,CTX_P['bootstrap']-start),ng))].sum(axis=1)
        draw=draw[(draw[:,0]>0)&(draw[:,2]>0)]
        r1=draw[:,1]/draw[:,0];r0=draw[:,3]/draw[:,2];global_r=(draw[:,1]+draw[:,3])/(draw[:,0]+draw[:,2])
        differences.extend(r1-r0);ok=global_r>0;lifts.extend(r1[ok]/global_r[ok])
    result['diff_low'],result['diff_high']=map(float,np.quantile(differences,[alpha/2,1-alpha/2]))
    result['lift_low'],result['lift_high']=map(float,np.quantile(lifts,[alpha/2,1-alpha/2]))
    return result


In [38]:
def ctx_new_test(raw):
    previous=core.clean_data(pd.concat([pd.read_csv(CTX_ROOT/'data/referencia_anterior/train.csv'),pd.read_csv(CTX_ROOT/'data/referencia_anterior/test.csv'),pd.read_csv(CTX_ROOT/'resultados/prueba_reservada.csv'),pd.read_csv(CTX_ROOT/'resultados/prueba_objetivo_popularidad.csv')],ignore_index=True))
    clean=core.clean_data(raw);used=core.artist_union(previous)
    mask=(~clean.track_id.isin(previous.track_id))&clean.artists.map(lambda s:bool(core.artists(s)) and not bool(core.artists(s)&used))
    mask &= ~np.isin(core.signatures(clean),core.signatures(previous));pool=clean.loc[mask]
    if len(pool)<CTX_P['new_evaluation_minimum']:raise ValueError('Pocas canciones adicionales disponibles con las exclusiones fijadas')
    test=pool.sample(n=min(len(pool),CTX_P['new_evaluation_size']),random_state=CTX_P['new_evaluation_seed']).reset_index(drop=True)
    assert not core.artist_union(previous)&core.artist_union(test)
    assert not set(previous.track_id)&set(test.track_id)
    assert not set(core.signatures(previous))&set(core.signatures(test))
    test.to_csv(CTX_OUT/'prueba_nueva.csv',index=False)
    return test,len(pool)


In [39]:
def ctx_report(base,frozen,comparison,effects,contexts,traits,stability,summary,counts):
    primary=summary['primary'];fine=effects.query('strategy=="fino" and configuration==@primary').iloc[0];baseline=effects.query('strategy=="referencia"').iloc[0]
    if fine.diff_low>0:verdict='El perfil fino candidato presenta mayor concentración en esta evaluación adicional, con un intervalo de diferencia completamente positivo.'
    elif fine.diff_high<0:verdict='La ventaja del perfil fino candidato se invierte en la evaluación adicional.'
    else:verdict='No se confirma una mayor concentración global del perfil fino candidato: su intervalo incluye cero.'
    supported=contexts.loc[(contexts.status=='ok')&(contexts.diff_low>0)] if len(contexts) else pd.DataFrame()
    new_result=dict(primary=primary,verdict=verdict,profile_rate=float(fine.profile_rate),global_rate=float(fine.global_rate),lift=float(fine.lift),difference_pp=float(100*fine.difference),difference_ci_pp=[float(100*fine.diff_low),float(100*fine.diff_high)],baseline_lift=float(baseline.lift),supported_contexts=supported.context.tolist() if len(supported) else [],context_claims_multiplicity_adjusted=True,n_new_evaluation=summary['n_new_evaluation'])
    blocks=['# ¿Existen perfiles sonoros que concentren canciones de alta popularidad y qué características los distinguen?',
        'Ampliación: perfiles más detallados y contraste por contextos de género. Se conserva la referencia anterior y la comparación K-Means/GMM antes y después de PCA.',
        '## Resumen actualizado',
        f'La configuración fina principal, {primary}, se eligió por calidad geométrica de validación sin consultar popularidad. El candidato identificado en entrenamiento presenta {fine.profile_rate:.2%} de canciones con popularidad ≥70 frente a {fine.global_rate:.2%} global en {summary["n_new_evaluation"]} canciones adicionales: lift {fine.lift:.2f}; diferencia frente al resto {100*fine.difference:+.2f} pp, IC95 [{100*fine.diff_low:+.2f}, {100*fine.diff_high:+.2f}]. {verdict}',
        f'La referencia de dos grupos presenta lift {baseline.lift:.2f} en estas mismas canciones. Los perfiles cubren conjuntos distintos: revisar cobertura y captura de canciones populares antes de interpretar una mayor concentración como mejora útil.',
        f'El candidato fino abarca {fine.coverage:.1%} de las canciones y contiene {fine.capture:.1%} de las canciones ≥70. Aun así, {1-fine.profile_rate:.1%} de sus canciones no alcanza ese umbral. Lift indica concentración relativa, no precisión predictiva.',
        '## 1. Ampliación y criterios fijados',
        'Hipótesis: dos grupos amplios pueden ocultar asociaciones dentro de perfiles más específicos o de contextos musicales. Se exploran K=3,4,5,6,8,10 para los dos algoritmos y espacios. Este brazo exige mayor granularidad deliberadamente; no se afirma que mejorar la popularidad requiera más grupos ni que su Silhouette deba superar al de K=2.',
        'Se conserva la preparación previamente congelada. Selección: Silhouette de validación en geometría común, grupos mínimos de 2 %, tolerancia 0,005 y preferencia por K menor y luego BIC. Modelo principal por mayor Silhouette de validación. El conjunto de validación ya fue observado en etapas previas; esta ampliación es exploratoria y usa otra evaluación adicional para contrastar candidatos.',
        'Solo después del ajuste se escoge el perfil de mayor tasa ≥70 en entrenamiento, con al menos 200 observaciones. Los contextos emplean un mínimo de 400 canciones de entrenamiento y 80 por perfil/resto. La lista de contextos y sus reglas se fijaron antes de observar sus tasas. No se seleccionan algoritmos, K o contextos por los resultados de prueba.',
        core.table(pd.DataFrame(frozen['selected'])[['model','space','k','covariance','val_sil_common']]),
        '## 2. Otra evaluación y comparación controlada',
        f'Se reservaron {summary["n_new_evaluation"]} canciones de {summary["eligible_pool"]} elegibles, excluyendo los 14.000 registros usados antes, sus nombres de artistas y los vectores acústicos exactos a siete decimales. El muestreo es uniforme, sin enriquecer artificialmente con canciones populares. Los nombres no son identificadores universales: alias pueden escapar de la exclusión.',
        'Tasa = canciones ≥70 / canciones con popularidad válida. Lift = tasa del perfil / tasa global del mismo conjunto. Cobertura = fracción del catálogo incluida en el perfil. Captura = fracción de todas las canciones ≥70 que están en él. No son precisión predictiva ni efectos causales.',
        core.table(effects),
        '![Concentración y cobertura en la nueva evaluación](ampliacion/figuras/concentracion.png)',
        'Las cuatro configuraciones finas comparten observaciones; solo la principal fija el contraste global primario. El resto es exploratorio. Los intervalos remuestrean componentes de artistas completos y son condicionales a los modelos ajustados, no a todo el pipeline.',
        'La diferencia entre lift fino y lift de referencia se presenta descriptivamente. El contraste con intervalo corresponde al candidato frente al resto de canciones, no a una prueba formal de superioridad entre ambos algoritmos.',
        '## 3. Contextos musicales: contraste posterior al clustering',
        'El CSV no contiene año, idioma ni estructura del audio. Se utilizan familias editoriales explícitas: electrónica, rock/metal, pop, hip-hop/R&B, latina y acústica/folk/country. Se recuperan todas las etiquetas brutas por ID, en lugar de conservar solo la primera. Una canción puede pertenecer a varios contextos: sus tamaños no se suman y no son confirmaciones independientes.',
        'Género no entra al vector acústico ni determina los hiperparámetros. Dentro de cada contexto elegible se escoge el candidato solo en entrenamiento, entre los perfiles del modelo fino principal. En prueba se exigen 150 canciones del contexto y 30 por candidato/resto; los casos insuficientes se reportan, no se interpretan como ausencia de asociación.',
        core.table(counts),core.table(contexts) if len(contexts) else 'No hubo contextos con tamaños de entrenamiento suficientes.',
        f'Los contextos secundarios utilizan intervalos bootstrap Bonferroni: alfa=0,05/{max(len(frozen["contexts"]),1)} según los contextos elegibles fijados en entrenamiento. La corrección es una aproximación por remuestreo; reduce hallazgos casuales al explorar varios contextos, pero no convierte esta revisión en un estudio causal o externo.',
        '![Contrastes por contexto con corrección múltiple](ampliacion/figuras/contextos.png)',
        ('Contextos con intervalo corregido completamente positivo: '+', '.join(supported.context)+'. Son asociaciones condicionadas a este diseño, no recetas de éxito.') if len(supported) else 'Ningún contexto evaluable conserva un intervalo corregido completamente positivo.',
        '### Sensibilidad descriptiva del candidato principal',core.table(pd.read_csv(CTX_OUT/'sensibilidad.csv')),
        'Se mantiene el candidato elegido con umbral 70. Los umbrales 60/80 son comprobaciones descriptivas, no nuevas oportunidades de seleccionar un modelo o declarar confirmación sin sus propios intervalos.',
        '## 4. Rasgos de los perfiles candidatos',core.table(traits),
        'Las medianas están en unidades originales. Las diferencias perfil−resto, estandarizadas por la DE de entrenamiento, describen qué distingue al grupo. No estiman cuánto subiría la popularidad al modificar energía, tempo u otro atributo.',
        '![Rasgos del candidato principal](ampliacion/figuras/rasgos.png)',
        '## 5. Calidad y estabilidad de la segmentación ampliada',core.table(comparison),
        'Las filas reference_original usan el mismo escalador de referencia y las mismas canciones para comparar modelos y PCA. Native usa la geometría de cada modelo. Métricas de contextos o muestras distintas no se comparan directamente. Ruido=0 es asignación completa, no ausencia comprobada de anomalías.',
        'Los tiempos de la ampliación corresponden al ajuste actual y a predicción de su nueva muestra; la referencia anterior predijo 2.000 canciones en otra ejecución. No comparar esos segundos como si fueran un benchmark con el mismo tamaño de prueba y carga del equipo.',
        core.table(pd.read_csv(CTX_OUT/'todos_los_perfiles.csv')),
        '![Mapa del modelo fino principal](ampliacion/figuras/mapa_principal.png)',
        'La proyección muestra PC1 y PC2; el clustering PCA utiliza todos los componentes retenidos. La separación visual no sustituye las métricas ni demuestra categorías musicales naturales.',
        core.table(stability.groupby(['model','space','kind']).ari.agg(['mean','min','max']).reset_index()),
        f'PCA de la ampliación conserva {summary["pca_components"]} componentes y {summary["pca_variance"]:.2%} de varianza. La preparación y PCA se ajustan solo con entrenamiento.',
        '## 6. Interpretación final y límites',verdict,
        'Un perfil más pequeño puede concentrar más canciones populares pero abarcar menos éxitos totales. La comparación informa concentración, cobertura, captura y separación; no se elige una historia mirando solo el mejor número. Si no mejora, el resultado se conserva.',
        'Persisten límites: popularidad histórica de conveniencia; pocas canciones ≥70; falta de exposición, promoción, fecha e idioma; atributos agregados sin melodía o estructura; candidatos seleccionados tras observar entrenamiento; intervalos condicionales y contextos solapados. No se afirma que no exista ningún patrón posible si este diseño no lo confirma.',
        '## 7. Referencia experimental y doce respuestas de la rúbrica']
    start=base.index('## 7. Experimento obligatorio de clustering y PCA');end=base.index('## 9. Cierre de la presentación')
    method=base[start:end].replace('## 7. Experimento obligatorio de clustering y PCA','### Experimento de referencia conservado')
    method=method.replace('### 12. Recomendación final','### 12. Recomendación final\n\nLa ampliación se interpreta con las secciones 2–6 anteriores. La recomendación y cifras de referencia siguientes corresponden al experimento anterior de 2.000 canciones, no a la nueva evaluación de la ampliación. Ninguna configuración se elige mirando popularidad de prueba.')
    blocks.extend(method.strip().split('\n\n'))
    blocks+=['## 9. Reproducción y presentación',
        'Ejecutar Proyecto_Mejorado.ipynb o python mejorar_proyecto.py para todas las etapas. Para recalcular esta ampliación desde resultados previos: python analizar_contextos.py. Protocolo, selección congelada, IDs, tablas y figuras están incluidos. El cierre auditivo es ilustrativo y requiere escuchar legalmente los enlaces, sin prometer un fragmento o estructura ganadora.',
        'Se utilizó ChatGPT/Codex para implementación y documentación. Completar autores y revisar personalmente antes de defender. Fuente: Pandya, M. (s. f.). Spotify Tracks Dataset [Conjunto de datos]. Hugging Face. https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset',
        'Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html']
    position=blocks.index('## 2. Otra evaluación y comparación controlada')
    blocks[position:position]=[f'![Búsqueda fina {family}/{space}](ampliacion/figuras/busqueda_{family}_{space}.png)' for family in ['KMeans','GMM'] for space in ['original','pca']]
    text='\n\n'.join(blocks);assert re.findall(r'^### (\d+)\. ',text,re.M)==[str(i) for i in range(1,13)]
    (CTX_ROOT/'resultados/informe.md').write_text(text,'utf-8');(CTX_ROOT/'resultados/informe.html').write_text(core.render_report(blocks),'utf-8')
    ctx_save('resultado_contextos.json',new_result)
    return new_result


In [40]:
def ejecutar_ampliacion():
    CTX_OUT.mkdir(exist_ok=True);(CTX_OUT/'figuras').mkdir(exist_ok=True)
    base=(CTX_ROOT/'resultados/informe.md').read_text('utf-8')
    if '## 1. Ampliación y criterios fijados' in base:
        pop.ejecutar_objetivo();base=(CTX_ROOT/'resultados/informe.md').read_text('utf-8')
    frozen_base=json.loads((CTX_ROOT/'resultados/seleccion_congelada.json').read_text('utf-8'))
    train=core.clean_data(pd.read_csv(CTX_ROOT/'resultados/entrenamiento.csv'))
    fit=core.clean_data(pd.read_csv(CTX_ROOT/'resultados/ajuste_interno.csv'));val=core.clean_data(pd.read_csv(CTX_ROOT/'resultados/validacion_interna.csv'))
    assert not core.artist_union(fit)&core.artist_union(val)
    selected=ctx_search(fit,val,frozen_base['preprocessing'])
    primary_row=pd.DataFrame(selected).sort_values(['val_sil_common','model','space'],ascending=[False,True,True]).iloc[0];primary=primary_row.model+'_'+primary_row.space
    prep=core.pipeline(frozen_base['preprocessing']).fit(train[core.FEATURES]);X=prep.transform(train[core.FEATURES]);pca=PCA(n_components=.9,svd_solver='full').fit(X)
    ref=core.pipeline('standard').fit(train[core.FEATURES]);common=ref.transform(train[core.FEATURES]);models={};train_labels={};leaders={}
    for cfg in selected:
        key=cfg['model']+'_'+cfg['space'];A=X if cfg['space']=='original' else pca.transform(X);e=core.estimator(cfg['model'],cfg['k'],cfg['covariance']);seconds,ok,_=core.fit_model(e,A)
        if not ok:raise RuntimeError('No converge ajuste final fino '+key)
        models[key]=(e,dict(cfg,final_fit_seconds=seconds));train_labels[key]=e.predict(A);leaders[key]=ctx_leader(train,train_labels[key],CTX_P['candidate_min_training'])
        if leaders[key] is None:raise RuntimeError('No hay candidato global elegible')
    baseline_decision=json.loads((CTX_ROOT/'resultados/perfiles_candidatos_congelados.json').read_text('utf-8'));baseline_key=baseline_decision['primary_model'];baseline_leader=baseline_decision['leaders'][baseline_key]
    bc=next(x for x in frozen_base['selected'] if x['model']+'_'+x['space']==baseline_key);baseline=core.estimator(bc['model'],bc['k'],bc['covariance']);A=X if bc['space']=='original' else pca.transform(X);_,ok,_=core.fit_model(baseline,A)
    if not ok:raise RuntimeError('No converge referencia')
    known=pd.read_csv(CTX_ROOT/f'resultados/asignaciones_{baseline_key}.csv').set_index('track_id')
    assert np.array_equal(baseline.predict(A),known.loc[train.track_id,'cluster'].to_numpy()),'Cambió la identidad de grupos de referencia'
    raw=pd.read_csv(CTX_ROOT/'data/raw/spotify_tracks.csv');training_contexts=ctx_tags(raw,[train])[0]
    context_leaders={};counts=[]
    for name,mask in training_contexts.items():
        leader=ctx_leader(train.loc[mask].reset_index(drop=True),train_labels[primary][mask],CTX_P['context_min_profile_training']) if mask.sum()>=CTX_P['context_min_training'] else None
        counts.append(dict(context=name,training_songs=int(mask.sum()),eligible=leader is not None))
        if leader is not None:context_leaders[name]=leader
    frozen=dict(primary=primary,selected=pd.DataFrame(selected).replace({np.nan:None}).to_dict('records'),leaders=leaders,contexts=context_leaders,baseline_key=baseline_key,baseline_leader=baseline_leader,preprocessing=frozen_base['preprocessing'],protocol_sha256=hashlib.sha256((CTX_ROOT/'protocolo_contextos.json').read_bytes()).hexdigest(),selection_uses_test=False,models_use_popularity=False,models_use_genre=False)
    ctx_save('seleccion_congelada.json',frozen)
    test,pool_size=ctx_new_test(raw);T=prep.transform(test[core.FEATURES]);common_test=ref.transform(test[core.FEATURES]);test_contexts=ctx_tags(raw,[test])[0]
    rows=[];stability=[];effects=[];traits=[];test_labels={};all_profiles=[]
    for key,(e,cfg) in models.items():
        A,B=(X,T) if cfg['space']=='original' else (pca.transform(X),pca.transform(T));lt=train_labels[key];tick=time.perf_counter();lv=e.predict(B);predict_seconds=time.perf_counter()-tick;test_labels[key]=lv
        pd.concat([train.assign(split='train',cluster=lt),test.assign(split='test',cluster=lv)]).to_csv(CTX_OUT/f'asignaciones_{key}.csv',index=False)
        for split,native,reference,labels in [('train',A,common,lt),('test',B,common_test,lv)]:
            ari=adjusted_rand_score(train_labels[cfg['model']+'_original'] if split=='train' else test_labels[cfg['model']+'_original'],labels) if cfg['space']=='pca' else np.nan
            for geometry,matrix in [('native',native),('reference_original',reference)]:rows.append(dict(model=cfg['model'],space=cfg['space'],split=split,evaluation_space=geometry,k=cfg['k'],covariance=cfg['covariance'],groups=len(np.unique(labels)),noise_pct=0.,fit_seconds=cfg['final_fit_seconds'],predict_seconds=predict_seconds,ari_vs_original=ari,**core.metrics(matrix,labels)))
        for kind in ['seed','subsample']:
            repeated=[]
            for seed in [42,7,21,123]:
                ix=np.random.default_rng(seed).choice(len(A),int(.8*len(A)),replace=False) if kind=='subsample' else np.arange(len(A))
                m=core.estimator(cfg['model'],cfg['k'],cfg['covariance'],seed);_,ok,_=core.fit_model(m,A[ix])
                if not ok:raise RuntimeError('No converge estabilidad '+key)
                repeated.append(m.predict(B))
            for i,j in itertools.combinations(range(4),2):stability.append(dict(model=cfg['model'],space=cfg['space'],kind=kind,ari=float(adjusted_rand_score(repeated[i],repeated[j]))))
        effect=ctx_effect(test,lv,leaders[key]);effects.append(dict(strategy='fino',configuration=key,cluster=leaders[key],**effect))
        traits.append(pop.pop_characteristics(train,lt,leaders[key],key))
        median=train.assign(cluster=lt).groupby('cluster')[core.FEATURES].median();median.to_csv(CTX_OUT/f'perfiles_{key}.csv')
        for cluster,r in median.iterrows():
            energy='intenso' if r.energy>=.65 else 'suave' if r.energy<.35 else 'intermedio';texture='acústico' if r.acousticness>=.6 else 'instrumental' if r.instrumentalness>=.5 else 'mixto'
            all_profiles.append(dict(configuration=key,cluster=int(cluster),n=int((lt==cluster).sum()),name=energy+' / '+texture,**r.to_dict()))
    baseline_labels=baseline.predict(T if bc['space']=='original' else pca.transform(T));effects.append(dict(strategy='referencia',configuration=baseline_key,cluster=baseline_leader,**ctx_effect(test,baseline_labels,baseline_leader)))
    context_rows=[];m=max(len(context_leaders),1)
    for name,leader in context_leaders.items():
        mask=test_contexts[name];frame=test.loc[mask].reset_index(drop=True);labels=test_labels[primary][mask]
        result=ctx_effect(frame,labels,leader,alpha=.05/m,min_side=CTX_P['context_min_profile_test']) if len(frame) else dict(status='muestra insuficiente',n=0,diff_low=np.nan,diff_high=np.nan,lift=np.nan)
        if len(frame)<CTX_P['context_min_test']:result['status']='muestra insuficiente';result['diff_low']=np.nan;result['diff_high']=np.nan
        context_rows.append(dict(context=name,cluster=leader,**result))
    comparison=pd.DataFrame(rows);effects=pd.DataFrame(effects);traits=pd.concat(traits,ignore_index=True);stability=pd.DataFrame(stability);contexts=pd.DataFrame(context_rows);counts=pd.DataFrame(counts)
    for filename,frame in [('comparativa',comparison),('concentracion',effects),('contextos',contexts),('rasgos',traits),('estabilidad',stability),('cobertura_contextos',counts)]:frame.to_csv(CTX_OUT/f'{filename}.csv',index=False)
    pd.DataFrame(all_profiles).to_csv(CTX_OUT/'todos_los_perfiles.csv',index=False)
    Z=pca.transform(X);plt.figure(figsize=(7,4));sc=plt.scatter(Z[:,0],Z[:,1],c=train_labels[primary],cmap='tab20',s=5,alpha=.4);plt.colorbar(sc,label='Grupo');plt.xlabel('PC1');plt.ylabel('PC2');plt.title('Modelo fino '+primary);ctx_figure('mapa_principal')
    plot=effects.copy();plt.figure(figsize=(9,4));plt.barh(plot.strategy+' / '+plot.configuration,100*plot.profile_rate,color='#358da0');plt.axvline(100*plot.global_rate.iloc[0],ls='--',color='black',label='Tasa global');plt.xlabel('Canciones ≥70 en el perfil (%)');plt.legend();plt.title('Concentración: comparar junto con cobertura y captura');ctx_figure('concentracion')
    plt.figure(figsize=(8,4))
    if len(contexts):
        for i,r in contexts.iterrows():
            if r.status=='ok':plt.plot([100*r.diff_low,100*r.diff_high],[i,i],color='#358da0');plt.scatter(100*r.difference,i,color='#123e50')
        plt.yticks(range(len(contexts)),contexts.context)
    plt.axvline(0,ls='--',color='black');plt.xlabel('Diferencia perfil−resto (pp), IC corregido');plt.title('Contextos secundarios');ctx_figure('contextos')
    q=traits.query('configuration==@primary').sort_values('standardized_difference');plt.figure(figsize=(8,4));plt.barh(q.variable,q.standardized_difference,color='#358da0');plt.xlabel('Diferencia de medianas / DE de entrenamiento');plt.axvline(0,color='black');plt.title('Rasgos del candidato fino principal');ctx_figure('rasgos')
    sensitivity=[]
    for threshold in CTX_P['sensitivity']:
        f=test.loc[test_labels[primary]==leaders[primary],'popularity'];base_rate=float((test.popularity>=threshold).mean());rate=float((f>=threshold).mean());sensitivity.append(dict(threshold=threshold,profile_rate=rate,global_rate=base_rate,lift=rate/base_rate if base_rate else np.nan))
    pd.DataFrame(sensitivity).to_csv(CTX_OUT/'sensibilidad.csv',index=False)
    summary=dict(primary=primary,n_new_evaluation=len(test),eligible_pool=pool_size,pca_components=int(pca.n_components_),pca_variance=float(pca.explained_variance_ratio_.sum()))
    result=ctx_report(base,frozen,comparison,effects,contexts,traits,stability,summary,counts)
    assert len(comparison)==16 and len(stability)==48 and not frozen['selection_uses_test']
    ctx_save('validacion.json',dict(status='passed',new_test=len(test),grid_candidates=36,comparison_rows=16,stability_pairs=48,context_count=len(context_leaders),no_previous_ids_artists_or_audio=True,protocol_sha256=frozen['protocol_sha256']))
    print(json.dumps(result,ensure_ascii=False,indent=2),flush=True)
    return result



In [ ]:
with threadpool_limits(limits=1):
    run()

## Respuesta a la pregunta principal

In [ ]:
print((OUT/'ampliacion/resultado_contextos.json').read_text('utf-8'))
print(pd.read_csv(OUT/'ampliacion/concentracion.csv').to_string(index=False))
print(pd.read_csv(OUT/'ampliacion/contextos.csv').to_string(index=False))

![Concentración ampliada](resultados/ampliacion/figuras/concentracion.png)

![Contextos e incertidumbre](resultados/ampliacion/figuras/contextos.png)

![Rasgos del candidato](resultados/ampliacion/figuras/rasgos.png)

In [ ]:
print((OUT/'informe.md').read_text('utf-8'))
print(pd.read_csv(OUT/'ampliacion/comparativa.csv').to_string(index=False))